# final notebook

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time
import warnings
import soccerdata as sd
from datetime import datetime
import re
import difflib
import unicodedata

## read each season data

In [ ]:
data_merged_gw_2016_17 = pd.read_csv('data/2016-17/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2017_18 = pd.read_csv('data/2017-18/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2018_19 = pd.read_csv('data/2018-19/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2019_20 = pd.read_csv('data/2019-20/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2020_21 = pd.read_csv('data/2020-21/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2021_22 = pd.read_csv('data/2021-22/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2022_23 = pd.read_csv('data/2022-23/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2023_24 = pd.read_csv('data/2023-24/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2024_25 = pd.read_csv('data/2024-25/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2025_26 = pd.read_csv('data/2025-26/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')

### note here that currently I am only making the actions not the analysis I did before to know what to do

## adding position column 

In [ ]:
# Add position column to merged gameweek data based on element_type from player raw data
def add_position_to_merged_gw(merged_gw_df, cleaned_players_df):
    # Map element_type to position names
    element_type_to_position = {
        1: 'Goalkeeper',
        2: 'Defender',
        3: 'Midfielder',
        4: 'Forward'
    }
    # create a new column 'position' in cleaned players dataframe
    cleaned_players_df['position'] = cleaned_players_df['element_type'].map(element_type_to_position)
    # create a mapping from player id to position
    player_id_to_position = dict(zip(cleaned_players_df['id'], cleaned_players_df['position']))
    # add the position column to the merged gw dataframe
    merged_gw_df['position'] = merged_gw_df['element'].map(player_id_to_position)
    return merged_gw_df


### load the players tables for the target seasons

In [ ]:
# first load the raw player data for each season until 2019-20
data_players_2016_17 = pd.read_csv('data/2016-17/players_raw.csv', encoding='latin-1', on_bad_lines='skip')
data_players_2017_18 = pd.read_csv('data/2017-18/players_raw.csv', encoding='latin-1', on_bad_lines='skip')
data_players_2018_19 = pd.read_csv('data/2018-19/players_raw.csv', encoding='latin-1', on_bad_lines='skip')
data_players_2019_20 = pd.read_csv('data/2019-20/players_raw.csv', encoding='latin-1', on_bad_lines='skip')

## add the position column

In [ ]:
# add the position column to each season's player data
data_merged_gw_2016_17 = add_position_to_merged_gw(data_merged_gw_2016_17, data_players_2016_17)
data_merged_gw_2017_18 = add_position_to_merged_gw(data_merged_gw_2017_18, data_players_2017_18)
data_merged_gw_2018_19 = add_position_to_merged_gw(data_merged_gw_2018_19, data_players_2018_19)
data_merged_gw_2019_20 = add_position_to_merged_gw(data_merged_gw_2019_20, data_players_2019_20)

## adding team name to the dataset

In [ ]:
# Add team names to seasons 2016-17 through 2019-20 using master team list
# Process:
# 1. Load master team list (contains season → team_id → team_name mapping)
# 2. Map player_id → team_id from player raw data
# 3. Map team_id → team_name from master list
# 4. Add team_name column to merged GW data
data_master_team_list = pd.read_csv('data/master_team_list.csv', encoding='latin-1', on_bad_lines='skip')
def add_team_name_to_merged_gw(merged_gw_df, players_raw_df, master_team_list_df, season):
    # filter the master team list for the given season
    season_team_list = master_team_list_df[master_team_list_df['season'] == season]
    # create a mapping from team id to team name
    team_id_to_name = dict(zip(season_team_list['team'], season_team_list['team_name']))
    # create a mapping from player id to team id
    player_id_to_team_id = dict(zip(players_raw_df['id'], players_raw_df['team']))
    # create a mapping from player id to team name
    player_id_to_team_name = {player_id: team_id_to_name.get(team_id, 'Unknown') for player_id, team_id in player_id_to_team_id.items()}

    # add the team name column to the merged gw dataframe    return merged_gw_df
    merged_gw_df['team'] = merged_gw_df['element'].map(player_id_to_team_name)

In [ ]:
# apply the function to each season's merged gw data
add_team_name_to_merged_gw(data_merged_gw_2016_17, data_players_2016_17, data_master_team_list, '2016-17')
add_team_name_to_merged_gw(data_merged_gw_2017_18, data_players_2017_18, data_master_team_list, '2017-18')
add_team_name_to_merged_gw(data_merged_gw_2018_19, data_players_2018_19, data_master_team_list, '2018-19')
add_team_name_to_merged_gw(data_merged_gw_2019_20, data_players_2019_20, data_master_team_list, '2019-20')

# ensuring all the seasons has the same attributes

In [ ]:
# Drop the xP (expected points) column from seasons 2020-21 through 2025-26 for consistency
data_merged_gw_2020_21 = data_merged_gw_2020_21.drop(columns=['xP'])
data_merged_gw_2021_22 = data_merged_gw_2021_22.drop(columns=['xP'])
data_merged_gw_2022_23 = data_merged_gw_2022_23.drop(columns=['xP'])
data_merged_gw_2023_24 = data_merged_gw_2023_24.drop(columns=['xP'])
data_merged_gw_2024_25 = data_merged_gw_2024_25.drop(columns=['xP'])
data_merged_gw_2025_26 = data_merged_gw_2025_26.drop(columns=['xP'])
# Drop columns not available across all seasons (2016-17 to 2018-19)
data_merged_gw_2016_17 = data_merged_gw_2016_17.drop(columns=['attempted_passes','big_chances_created','big_chances_missed','completed_passes','dribbles','ea_index','errors_leading_to_goal','errors_leading_to_goal_attempt','fouls','key_passes','kickoff_time_formatted','loaned_in','loaned_out','offside','open_play_crosses','penalties_conceded','tackled','target_missed','winning_goals'])
data_merged_gw_2017_18 = data_merged_gw_2017_18.drop(columns=['attempted_passes','big_chances_created','big_chances_missed','completed_passes','dribbles','ea_index','errors_leading_to_goal','errors_leading_to_goal_attempt','fouls','key_passes','kickoff_time_formatted','loaned_in','loaned_out','offside','open_play_crosses','penalties_conceded','tackled','target_missed','winning_goals'])
data_merged_gw_2018_19 = data_merged_gw_2018_19.drop(columns=['attempted_passes','big_chances_created','big_chances_missed','completed_passes','dribbles','ea_index','errors_leading_to_goal','errors_leading_to_goal_attempt','fouls','key_passes','kickoff_time_formatted','loaned_in','loaned_out','offside','open_play_crosses','penalties_conceded','tackled','target_missed','winning_goals'])
# Drop from 2022 the xp and that stuff
data_merged_gw_2022_23 = data_merged_gw_2022_23.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
data_merged_gw_2023_24 = data_merged_gw_2023_24.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
data_merged_gw_2024_25 = data_merged_gw_2024_25.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
data_merged_gw_2025_26 = data_merged_gw_2025_26.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
# Drop modified in the last two seasons
data_merged_gw_2024_25 = data_merged_gw_2024_25.drop(columns=['modified'])
data_merged_gw_2025_26 = data_merged_gw_2025_26.drop(columns=['modified'])
# drop the id from the first two seasons
data_merged_gw_2016_17 = data_merged_gw_2016_17.drop(columns=['id'])
data_merged_gw_2017_18 = data_merged_gw_2017_18.drop(columns=['id'])
data_merged_gw_2018_19 = data_merged_gw_2018_19.drop(columns=['id'])

## adding defensive contribution

In [ ]:
# Calculate defensive_contribution for seasons 2016-17 to 2024-25
# FPL scoring rules:
# - Defenders: clearances_blocks_interceptions + tackles
# - Midfielders/Forwards: clearances_blocks_interceptions + tackles + recoveries
def add_defensive_contribution(merged_gw_df):
    def calculate_defensive_contribution(row):
        position = row['position']
        clearances = row.get('clearances_blocks_interceptions', 0)
        tackles = row.get('tackles', 0)
        recoveries = row.get('recoveries', 0)
        if position == 'Defender':
            return clearances + tackles
        elif position in ['Midfielder', 'Forward']:
            return clearances + tackles + recoveries
        else:
            return 0
    merged_gw_df['defensive_contribution'] = merged_gw_df.apply(calculate_defensive_contribution, axis=1)
    return merged_gw_df

In [ ]:
# Add placeholder columns (value 0) for defensive stats not tracked in seasons 2019-20 to 2024-25
# This ensures consistent schema across all seasons before calculating defensive_contribution
data_merged_gw_2019_20['clearances_blocks_interceptions'] = 0
data_merged_gw_2019_20['recoveries'] = 0
data_merged_gw_2019_20['tackles'] = 0
data_merged_gw_2020_21['clearances_blocks_interceptions'] = 0
data_merged_gw_2020_21['recoveries'] = 0
data_merged_gw_2020_21['tackles'] = 0
data_merged_gw_2021_22['clearances_blocks_interceptions'] = 0
data_merged_gw_2021_22['recoveries'] = 0
data_merged_gw_2021_22['tackles'] = 0
data_merged_gw_2022_23['clearances_blocks_interceptions'] = 0
data_merged_gw_2022_23['recoveries'] = 0
data_merged_gw_2022_23['tackles'] = 0
data_merged_gw_2023_24['clearances_blocks_interceptions'] = 0
data_merged_gw_2023_24['recoveries'] = 0
data_merged_gw_2023_24['tackles'] = 0
data_merged_gw_2024_25['clearances_blocks_interceptions'] = 0
data_merged_gw_2024_25['recoveries'] = 0
data_merged_gw_2024_25['tackles'] = 0
data_merged_gw_2016_17 = add_defensive_contribution(data_merged_gw_2016_17)
data_merged_gw_2017_18 = add_defensive_contribution(data_merged_gw_2017_18)
data_merged_gw_2018_19 = add_defensive_contribution(data_merged_gw_2018_19)
data_merged_gw_2019_20 = add_defensive_contribution(data_merged_gw_2019_20)
data_merged_gw_2020_21 = add_defensive_contribution(data_merged_gw_2020_21)
data_merged_gw_2021_22 = add_defensive_contribution(data_merged_gw_2021_22)
data_merged_gw_2022_23 = add_defensive_contribution(data_merged_gw_2022_23)
data_merged_gw_2023_24 = add_defensive_contribution(data_merged_gw_2023_24)
data_merged_gw_2024_25 = add_defensive_contribution(data_merged_gw_2024_25)

### verify that all the cols are identical now

In [ ]:
# compare the columns of all these datasets
merged_2016_17_columns = set(data_merged_gw_2016_17.columns.tolist())
merged_2017_18_columns = set(data_merged_gw_2017_18.columns.tolist())
merged_2018_19_columns = set(data_merged_gw_2018_19.columns.tolist())
merged_2019_20_columns = set(data_merged_gw_2019_20.columns.tolist())
merged_2020_21_columns = set(data_merged_gw_2020_21.columns.tolist())
merged_2021_22_columns = set(data_merged_gw_2021_22.columns.tolist())
merged_2022_23_columns = set(data_merged_gw_2022_23.columns.tolist())
merged_2023_24_columns = set(data_merged_gw_2023_24.columns.tolist())
merged_2024_25_columns = set(data_merged_gw_2024_25.columns.tolist())
merged_2025_26_columns = set(data_merged_gw_2025_26.columns.tolist())
# find the common columns across all seasons
common_merged_columns_all_seasons = merged_2016_17_columns.intersection(merged_2017_18_columns).intersection(merged_2018_19_columns).intersection(merged_2019_20_columns).intersection(merged_2020_21_columns).intersection(merged_2021_22_columns).intersection(merged_2022_23_columns).intersection(merged_2023_24_columns).intersection(merged_2024_25_columns).intersection(merged_2025_26_columns)
print("Common Columns Across All Seasons:", sorted(common_merged_columns_all_seasons))
# find the unique columns in each season compared to the common columns
unique_2016_17_columns = merged_2016_17_columns - common_merged_columns_all_seasons
unique_2017_18_columns = merged_2017_18_columns - common_merged_columns_all_seasons
unique_2018_19_columns = merged_2018_19_columns - common_merged_columns_all_seasons
unique_2019_20_columns = merged_2019_20_columns - common_merged_columns_all_seasons
unique_2020_21_columns = merged_2020_21_columns - common_merged_columns_all_seasons
unique_2021_22_columns = merged_2021_22_columns - common_merged_columns_all_seasons
unique_2022_23_columns = merged_2022_23_columns - common_merged_columns_all_seasons
unique_2023_24_columns = merged_2023_24_columns - common_merged_columns_all_seasons
unique_2024_25_columns = merged_2024_25_columns - common_merged_columns_all_seasons
unique_2025_26_columns = merged_2025_26_columns - common_merged_columns_all_seasons
print("Unique Columns in 2016-17:", sorted(unique_2016_17_columns))
print("Unique Columns in 2017-18:", sorted(unique_2017_18_columns))
print("Unique Columns in 2018-19:", sorted(unique_2018_19_columns))
print("Unique Columns in 2019-20:", sorted(unique_2019_20_columns))
print("Unique Columns in 2020-21:", sorted(unique_2020_21_columns))
print("Unique Columns in 2021-22:", sorted(unique_2021_22_columns))
print("Unique Columns in 2022-23:", sorted(unique_2022_23_columns))
print("Unique Columns in 2023-24:", sorted(unique_2023_24_columns))
print("Unique Columns in 2024-25:", sorted(unique_2024_25_columns))
print("Unique Columns in 2025-26:", sorted(unique_2025_26_columns))


# hsitorical points adjustment

## here there is a key decision: 
### the points of the previous seasons will be modified to work with the same system as the new rules (adding points for defensive contribution)

### consider moving this to the end (after merging with the defensive data)

1. Standardize position values across all seasons2. Adjust points for seasons 2016-17 to 2018-19 to match current FPL defensive contribution scoring

In [ ]:
# Adjust historical points (2016-17 to 2018-19) to align with current FPL scoring system
# Add 2 bonus points when:
# - Defenders reach defensive_contribution >= 10
# - Midfielders/Forwards reach defensive_contribution >= 12
def modify_points(merged_gw_df):
    def calculate_modified_points(row):
        points = row['total_points']
        position = row['position']
        defensive_contribution = row['defensive_contribution']
        if position == 'Defender' and defensive_contribution >= 10:
            points += 2
        elif position in ['Midfielder', 'Forward'] and defensive_contribution >= 12:
            points += 2
        return points
    merged_gw_df['total_points'] = merged_gw_df.apply(calculate_modified_points, axis=1)
    return merged_gw_df
data_merged_gw_2016_17 = modify_points(data_merged_gw_2016_17)
data_merged_gw_2017_18 = modify_points(data_merged_gw_2017_18)
data_merged_gw_2018_19 = modify_points(data_merged_gw_2018_19)

### note here that you need to run modify points of the seasons from 2019 to 2025 when you merge with the defensive data

# merging all seasons data in a single dataset

In [ ]:
# Add season identifier to each dataset before merging
data_merged_gw_2016_17['season'] = '2016-17'
data_merged_gw_2017_18['season'] = '2017-18'
data_merged_gw_2018_19['season'] = '2018-19'
data_merged_gw_2019_20['season'] = '2019-20'
data_merged_gw_2020_21['season'] = '2020-21'
data_merged_gw_2021_22['season'] = '2021-22'
data_merged_gw_2022_23['season'] = '2022-23'
data_merged_gw_2023_24['season'] = '2023-24'
data_merged_gw_2024_25['season'] = '2024-25'
data_merged_gw_2025_26['season'] = '2025-26'
# concatenating all seasons data into a single dataframe
all_seasons_data = pd.concat([data_merged_gw_2016_17, data_merged_gw_2017_18, data_merged_gw_2018_19, data_merged_gw_2019_20, data_merged_gw_2020_21, data_merged_gw_2021_22, data_merged_gw_2022_23, data_merged_gw_2023_24, data_merged_gw_2024_25, data_merged_gw_2025_26], ignore_index=True)
print("All Seasons Data Sample:")
print(all_seasons_data.sample(10))

## standarizing the position across seasons

In [ ]:
# Standardize position values to short codes for consistency
all_seasons_data['position'] = all_seasons_data['position'].replace({'Goalkeeper': 'GK', 'Defender': 'DEF', 'Midfielder': 'MID', 'Forward': 'FWD'})

## converting the opponent team from id to name

In [ ]:
# Convert opponent_team from team IDs to team names using master team list
# Create season-specific mappings of team_id -> team_name
team_id_name_mapping = {}
for season in all_seasons_data['season'].unique():
    season_team_data = data_master_team_list[data_master_team_list['season'] == season]
    team_id_name_mapping[season] = dict(zip(season_team_data['team'], season_team_data['team_name']))
# replace the opponent_team in all_seasons_data based on the season and the team id
def replace_opponent_team(row):
    season = row['season']
    team_id = row['opponent_team']
    return team_id_name_mapping[season].get(team_id, team_id)
all_seasons_data['opponent_team'] = all_seasons_data.apply(replace_opponent_team, axis=1)

# saving the current state of data as csv

In [ ]:
# save all seasons data to a csv file
all_seasons_data.to_csv('all_seasons_data.csv', index=False, encoding='latin-1')

# here we add the game number feature then we make the merging with defensive stats

## here I will add game_number to both datasets

## Load Datasets

In [ ]:
# Load the datasets
print("Loading all_seasons_data.csv...")
all_seasons_df = pd.read_csv('all_seasons_data.csv')

print("Loading defensive_stats_raw.csv...")
defensive_stats_df = pd.read_csv('defensive_stats_raw.csv')

print(f"\n✓ All seasons data shape: {all_seasons_df.shape}")
print(f"✓ Defensive stats shape: {defensive_stats_df.shape}")

print(f"\nSeasons in all_seasons_data: {sorted(all_seasons_df['season'].unique())}")
print(f"Seasons in defensive_stats: {sorted(defensive_stats_df['season'].unique())}")

## Step 1: Build Team-Game Mapping from fixtures.csv

For each season (2018-19 to 2024-25):
1. Load fixtures.csv and teams.csv
2. Create team_id → team_name mapping
3. Convert each fixture to 2 team-game records (home + away)
4. Sort chronologically and assign game_number 1-38

In [ ]:
def build_game_number_mapping():
    """
    Build a comprehensive mapping of (season, fixture_id) → game_number
    using fixtures.csv as the source of truth.
    
    Only processes seasons that have BOTH fixtures.csv AND teams.csv to ensure
    accurate team ID to name mapping.
    
    Returns:
        - fixture_to_game_number: DataFrame with (season, fixture_id, team_name, game_number)
    """
    
    # Season folders to process (only those with teams.csv)
    season_folders = ['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
    
    all_mappings = []
    
    for season in season_folders:
        fixtures_path = f'data/{season}/fixtures.csv'
        teams_path = f'data/{season}/teams.csv'
        
        try:
            # Load fixtures and teams
            fixtures_df = pd.read_csv(fixtures_path)
            teams_df = pd.read_csv(teams_path)
            
            # Create team_id → team_name mapping
            team_id_to_name = dict(zip(teams_df['id'], teams_df['name']))
            
            # Convert kickoff_time to datetime
            fixtures_df['kickoff_time'] = pd.to_datetime(fixtures_df['kickoff_time'])
            
            # Keep only finished matches
            finished_fixtures = fixtures_df[fixtures_df['finished'] == True].copy()
            
            if len(finished_fixtures) == 0:
                print(f"{season}: No finished matches found, skipping")
                continue
            
            # Create team-game records (2 per fixture: home + away)
            team_games = []
            
            for _, fixture in finished_fixtures.iterrows():
                fixture_id = fixture['id']
                kickoff_time = fixture['kickoff_time']
                gw = fixture['event']
                
                # Home team record
                team_games.append({
                    'season': season,
                    'fixture_id': fixture_id,
                    'team_id': fixture['team_h'],
                    'team_name': team_id_to_name.get(fixture['team_h'], 'Unknown'),
                    'kickoff_time': kickoff_time,
                    'gw': gw,
                    'is_home': True
                })
                
                # Away team record
                team_games.append({
                    'season': season,
                    'fixture_id': fixture_id,
                    'team_id': fixture['team_a'],
                    'team_name': team_id_to_name.get(fixture['team_a'], 'Unknown'),
                    'kickoff_time': kickoff_time,
                    'gw': gw,
                    'is_home': False
                })
            
            season_df = pd.DataFrame(team_games)
            
            # Sort by team and kickoff_time (chronological order)
            season_df = season_df.sort_values(['team_name', 'kickoff_time'])
            
            # Assign game_number per team (1, 2, 3, ..., up to 38)
            season_df['game_number'] = season_df.groupby('team_name').cumcount() + 1
            
            all_mappings.append(season_df)
            
            print(f"{season}: {len(finished_fixtures)} fixtures → {len(season_df)} team-game records")
            print(f"         Teams: {season_df['team_name'].nunique()}, Max game_number: {season_df['game_number'].max()}")
            
        except FileNotFoundError as e:
            print(f"{season}: Required file not found, skipping ({e})")
        except Exception as e:
            print(f"{season}: Error - {e}")
    
    # Combine all seasons
    if all_mappings:
        full_mapping = pd.concat(all_mappings, ignore_index=True)
        print(f"\n✅ Total mapping records: {len(full_mapping):,}")
        print(f"Note: Seasons 2016-17, 2017-18, 2018-19 excluded (missing teams.csv)")
        return full_mapping
    else:
        print("❌ No mappings created!")
        return None

# Build the mapping
print("="*80)
print("STEP 1: Building game_number mapping from fixtures.csv")
print("="*80)
game_number_mapping = build_game_number_mapping()

## Step 2: Join game_number to all_seasons_data.csv

Join using the fixture ID (100% match rate for seasons with fixtures.csv)

In [ ]:
def add_game_number_to_all_seasons(df, mapping):
    """
    Add game_number to all_seasons_data using fixture ID matching.
    
    Args:
        df: all_seasons_data DataFrame
        mapping: game_number_mapping DataFrame from Step 1
        
    Returns:
        DataFrame with game_number column added
    """
    df = df.copy()
    
    # Drop existing game_number if present
    if 'game_number' in df.columns:
        df = df.drop('game_number', axis=1)
        print("Dropped existing game_number column")
    
    # Create slim mapping for joining: (season, fixture_id, team_name) → game_number
    # We need team_name because each fixture has 2 teams
    slim_mapping = mapping[['season', 'fixture_id', 'team_name', 'game_number']].copy()
    slim_mapping = slim_mapping.rename(columns={'fixture_id': 'fixture', 'team_name': 'team'})
    
    print(f"Original records: {len(df):,}")
    print(f"Mapping records: {len(slim_mapping):,}")
    
    # Merge on season, fixture, and team
    df = df.merge(
        slim_mapping,
        on=['season', 'fixture', 'team'],
        how='left'
    )
    
    # Convert to nullable integer
    df['game_number'] = df['game_number'].astype('Int64')
    
    # Report results
    matched = df['game_number'].notna().sum()
    unmatched = df['game_number'].isna().sum()
    
    print(f"\n✅ Matched records: {matched:,} ({matched/len(df)*100:.1f}%)")
    print(f"❌ Unmatched records: {unmatched:,} ({unmatched/len(df)*100:.1f}%)")
    
    # Check which seasons have unmatched records
    if unmatched > 0:
        unmatched_by_season = df[df['game_number'].isna()].groupby('season').size()
        print(f"\nUnmatched by season:")
        print(unmatched_by_season)
    
    return df

# Apply to all_seasons_data
print("="*80)
print("STEP 2: Adding game_number to all_seasons_data.csv")
print("="*80)
all_seasons_updated = add_game_number_to_all_seasons(all_seasons_df, game_number_mapping)

## Step 3: Add game_number to defensive_stats_raw.csv

Since defensive_stats doesn't have fixture IDs, we'll use date-based matching:
1. Parse the 'game' column to extract date and teams
2. Match with fixtures using date + team combination

In [ ]:
def add_game_number_to_defensive_stats(df, mapping):
    """
    Add game_number to defensive_stats using date-based matching.
    
    The 'game' column format: "YYYY-MM-DD Team1-Team2"
    The 'season' column format: "1920" (for 2019-20)
    
    Args:
        df: defensive_stats DataFrame
        mapping: game_number_mapping DataFrame
        
    Returns:
        DataFrame with game_number column added
    """
    df = df.copy()
    
    # Drop existing game_number if present
    if 'game_number' in df.columns:
        df = df.drop('game_number', axis=1)
        print("Dropped existing game_number column")
    
    # Team name mapping: defensive_stats name → mapping name (FPL short name)
    team_name_mapping = {
        'Brighton & Hove Albion': 'Brighton',
        'Ipswich Town': 'Ipswich',
        'Leeds United': 'Leeds',
        'Leicester City': 'Leicester',
        'Luton Town': 'Luton',
        'Manchester City': 'Man City',
        'Manchester United': 'Man Utd',
        'Newcastle United': 'Newcastle',
        'Norwich City': 'Norwich',
        'Nottingham Forest': "Nott'm Forest",
        'Sheffield United': 'Sheffield Utd',
        'Tottenham Hotspur': 'Spurs',
        'West Bromwich Albion': 'West Brom',
        'West Ham United': 'West Ham',
        'Wolverhampton Wanderers': 'Wolves',
    }
    
    # Map team names to match FPL naming
    df['team_mapped'] = df['team'].map(team_name_mapping).fillna(df['team'])
    
    # Extract date from 'game' column (format: "YYYY-MM-DD Team1-Team2")
    df['game_date'] = pd.to_datetime(df['game'].str.extract(r'^(\d{4}-\d{2}-\d{2})')[0], errors='coerce')
    
    # Convert defensive stats season format (1920) to standard format (2019-20)
    def convert_season(s):
        if pd.isna(s):
            return None
        s = str(s).replace('.0', '')
        if len(s) == 4:  # e.g., "1920"
            return f"20{s[:2]}-{s[2:]}"
        return s
    
    df['season_standard'] = df['season'].apply(convert_season)
    
    # Create date-based mapping from fixtures
    mapping_for_date = mapping.copy()
    mapping_for_date['game_date'] = mapping_for_date['kickoff_time'].dt.date
    mapping_for_date['game_date'] = pd.to_datetime(mapping_for_date['game_date'])
    
    # Create slim mapping: (season, team_name, game_date) → game_number
    date_mapping = mapping_for_date[['season', 'team_name', 'game_date', 'game_number']].copy()
    date_mapping = date_mapping.rename(columns={'team_name': 'team_mapped', 'season': 'season_standard'})
    
    # Remove duplicates (same team can't have 2 games on same day)
    date_mapping = date_mapping.drop_duplicates(subset=['season_standard', 'team_mapped', 'game_date'])
    
    print(f"Original records: {len(df):,}")
    print(f"Date mapping records: {len(date_mapping):,}")
    
    # Merge
    df = df.merge(
        date_mapping,
        on=['season_standard', 'team_mapped', 'game_date'],
        how='left'
    )
    
    # Convert to nullable integer
    df['game_number'] = df['game_number'].astype('Int64')
    
    # Clean up temporary columns
    df = df.drop(['game_date', 'season_standard', 'team_mapped'], axis=1)
    
    # Report results
    matched = df['game_number'].notna().sum()
    unmatched = df['game_number'].isna().sum()
    
    print(f"\n✅ Matched records: {matched:,} ({matched/len(df)*100:.1f}%)")
    print(f"❌ Unmatched records: {unmatched:,} ({unmatched/len(df)*100:.1f}%)")
    
    # Check which seasons have unmatched records
    if unmatched > 0:
        unmatched_by_season = df[df['game_number'].isna()].groupby('season').size()
        print(f"\nUnmatched by season:")
        print(unmatched_by_season)
    
    return df

# Apply to defensive_stats
print("="*80)
print("STEP 3: Adding game_number to defensive_stats_raw.csv")
print("="*80)
defensive_stats_updated = add_game_number_to_defensive_stats(defensive_stats_df, game_number_mapping)

## Step 5: Save Updated Datasets

In [ ]:
# Save the updated datasets
print("Saving updated datasets...")
print("="*80)

# Save all_seasons_data with game_number
all_seasons_updated.to_csv('all_seasons_data.csv', index=False)
print(f"✓ Saved: all_seasons_data.csv")
print(f"  - {len(all_seasons_updated):,} records")
print(f"  - game_number range: 1-{all_seasons_updated['game_number'].max()}")

# Save defensive_stats with game_number
defensive_stats_updated.to_csv('defensive_stats_raw.csv', index=False)
print(f"\n✓ Saved: defensive_stats_raw.csv")
print(f"  - {len(defensive_stats_updated):,} records")
print(f"  - game_number range: 1-{defensive_stats_updated['game_number'].max()}")

print("\n" + "="*80)
print("ALL DONE!")
print("="*80)

# now merging the main table with the defensive stats table

### read both tables

In [ ]:
# Load the raw defensive data
print("Loading defensive stats raw data...")
df_def = pd.read_csv('defensive_stats_raw.csv', low_memory=False)
print(f"Original defensive data shape: {df_def.shape}")

# Remove the header row (row 0 contains column descriptions)
df_def = df_def[df_def['season'].notna() & (df_def['season'] != '')]
df_def = df_def.reset_index(drop=True)
print(f"After removing header row: {df_def.shape}")

In [ ]:
# 1. STANDARDIZE SEASON FORMAT (1920 -> 2019-20)
def convert_season_format(season_code):
    """Convert season from '1920' to '2019-20' format"""
    if pd.isna(season_code) or season_code == '':
        return None
    try:
        season_str = str(int(float(season_code)))
        if len(season_str) == 4:
            year1 = int('20' + season_str[:2])
            year2 = season_str[2:]
            return f"{year1}-{year2}"
        return None
    except:
        return None

df_def['season'] = df_def['season'].apply(convert_season_format)
print(f"\nSeasons after conversion:")
print(df_def['season'].value_counts().sort_index())

In [ ]:
# 2. COMBINE TACKLE COLUMNS (Def 3rd, Mid 3rd, Att 3rd -> Total Tackles)
print("="*60)
print("COMBINING TACKLE COLUMNS")
print("="*60)

# The columns are named 'Tackles.2' (Def 3rd), 'Tackles.3' (Mid 3rd), 'Tackles.4' (Att 3rd)
# Convert to numeric and combine
tackle_cols = ['Tackles.2', 'Tackles.3', 'Tackles.4']
for col in tackle_cols:
    df_def[col] = pd.to_numeric(df_def[col], errors='coerce')

# Create combined tackles column (sum of all three thirds)
df_def['tackles_total'] = df_def[tackle_cols].sum(axis=1)

print(f"✓ Created 'tackles_total' by combining tackles from all thirds")
print(f"  Sample values: {df_def['tackles_total'].head(10).tolist()}")

In [ ]:
# 2. COMBINE TACKLE COLUMNS (Def 3rd, Mid 3rd, Att 3rd -> Total Tackles)
# The columns are named 'Tackles.2', 'Tackles.3', 'Tackles.4' representing Def 3rd, Mid 3rd, Att 3rd
print("="*60)
print("COMBINING TACKLE COLUMNS")
print("="*60)

# Convert tackle columns to numeric
tackle_cols = ['Tackles.2', 'Tackles.3', 'Tackles.4']  # Def 3rd, Mid 3rd, Att 3rd
for col in tackle_cols:
    df_def[col] = pd.to_numeric(df_def[col], errors='coerce')

# Create combined tackles column (sum of all three thirds)
df_def['tackles_total'] = df_def[tackle_cols].sum(axis=1)

print(f"Created 'tackles_total' column by combining:")
print(f"  - Tackles.2 (Def 3rd)")
print(f"  - Tackles.3 (Mid 3rd)")
print(f"  - Tackles.4 (Att 3rd)")
print(f"\nSample: {df_def['tackles_total'].describe()}")

In [ ]:
# 3. SELECT AND RENAME DEFENSIVE COLUMNS
print("="*60)
print("SELECTING DEFENSIVE COLUMNS")
print("="*60)

# Map raw columns to FPL-style naming
column_mapping = {
    'season': 'season',
    'player': 'name',
    'team': 'team',
    'pos': 'position',
    'min': 'minutes',
    'Tackles': 'tackles',  # Total tackles (Tkl)
    'Tackles.1': 'tackles_won',  # TklW
    'tackles_total': 'tackles_total',  # Combined Def+Mid+Att third
    'Challenges': 'challenges',  # Total challenges
    'Challenges.1': 'challenges_attempted',  # Att
    'Challenges.2': 'challenges_success_rate',  # Tkl%
    'Challenges.3': 'challenges_lost',  # Lost
    'Blocks': 'blocks',  # Total blocks
    'Blocks.1': 'blocks_shots',  # Sh
    'Blocks.2': 'blocks_passes',  # Pass
    'Int': 'interceptions',  # Interceptions
    'Tkl+Int': 'tackles_interceptions',  # Tkl+Int
    'Clr': 'clearances',  # Clearances
    'Err': 'errors',  # Errors leading to shots
    'match_id': 'match_id',
    'game': 'game'
}

# Select only the columns we need for defensive stats
defensive_columns = list(column_mapping.keys())
df_def_selected = df_def[defensive_columns].copy()

# Rename columns to match FPL naming
df_def_selected = df_def_selected.rename(columns=column_mapping)

print(f"Selected columns: {df_def_selected.columns.tolist()}")

In [ ]:
# 4. CONVERT DATA TYPES AND FILL MISSING VALUES
print("="*60)
print("CONVERTING DATA TYPES")
print("="*60)

# Numeric columns
numeric_cols = [
    'minutes', 'tackles', 'tackles_won', 'tackles_total',
    'challenges', 'challenges_attempted', 'challenges_success_rate', 
    'challenges_lost', 'blocks', 'blocks_shots', 'blocks_passes',
    'interceptions', 'tackles_interceptions', 'clearances', 'errors'
]

for col in numeric_cols:
    df_def_selected[col] = pd.to_numeric(df_def_selected[col], errors='coerce')

# Fill NaN values with 0 for defensive stats (no stat = 0)
df_def_selected[numeric_cols] = df_def_selected[numeric_cols].fillna(0)

print("✓ Converted numeric columns and filled NaN with 0")

# Assign gameweek

In [ ]:
# 5. EXTRACT GAMEWEEK FROM MATCH DATA
print("="*60)
print("EXTRACTING GAMEWEEK INFORMATION")
print("="*60)

# Parse the 'game' column to extract date (Format: "2019-08-09 Liverpool-Norwich City")
def extract_date(game_str):
    """Extract date from game string"""
    if pd.isna(game_str) or game_str == '':
        return None
    try:
        parts = str(game_str).split(' ')
        if len(parts) > 0:
            return parts[0]  # Return the date part
    except:
        pass
    return None

df_def_selected['game_date'] = df_def_selected['game'].apply(extract_date)
df_def_selected['game_date'] = pd.to_datetime(df_def_selected['game_date'], errors='coerce')

# Assign gameweek based on season and date
def assign_gameweek(row):
    """Assign gameweek based on season and date"""
    if pd.isna(row['game_date']) or pd.isna(row['season']):
        return None
    
    season = row['season']
    game_date = row['game_date']
    
    try:
        year = int(season.split('-')[0])
    except:
        return None
    
    # Season typically starts in early August
    season_start = pd.Timestamp(f'{year}-08-01')
    
    # Calculate weeks from season start
    weeks_diff = (game_date - season_start).days // 7
    
    # Gameweek is approximately weeks + 1, capped at 38
    gw = min(max(weeks_diff + 1, 1), 38)
    
    return int(gw)

df_def_selected['GW'] = df_def_selected.apply(assign_gameweek, axis=1)

print(f"✓ Assigned gameweeks. GW range: {df_def_selected['GW'].min()} to {df_def_selected['GW'].max()}")

## save the cleaned data into csv

In [ ]:
df_def_selected.to_csv("defensive_stats_cleaned.csv", index=False)

# Now the merging part:

### load both datasets

In [ ]:
# loading both datasets
df_def = pd.read_csv("defensive_stats_cleaned.csv")
df_main = pd.read_csv("all_seasons_data.csv")

### cleaning the name field 

In [ ]:

# ==========================================
# STEP 1: FORCE-CLEAN NAMES
# ==========================================

def clean_main_name_format(name):
    if not isinstance(name, str):
        return str(name)
    
    # 1. Fix Mojibake (Encoding Errors)
    char_map = {
        'Ã©': 'é', 'Ãº': 'ú', 'Ã¡': 'á', 'Ã³': 'ó', 'Ã¨': 'è', 'Ã±': 'ñ',
        'Ã\xad': 'í', 'Ã§': 'ç', 'Ã¢': 'â', 'Ã¼': 'ü', 'Ã¶': 'ö', 'Ã\x9f': 'ß',
        'Ã¸': 'ø', 'Ã«': 'ë', 'Ã': 'à' ,'à£': 'ã', 'à©': 'é'
    }
    for bad, good in char_map.items():
        name = name.replace(bad, good)

    # 2. Remove trailing IDs (e.g., '_376', '_12', '_4')
    # Regex: Underscore followed by 1 or more digits at the END of the string
    name = re.sub(r'_\d+$', '', name)
    
    # 3. Replace remaining underscores with spaces
    name = name.replace('_', ' ')
    
    # 4. Standardize (lower, strip)
    return name.lower().strip()

# Create/Overwrite the 'join_name' column
df_main['join_name'] = df_main['name'].apply(clean_main_name_format)


### filter only needed seasons

In [ ]:
# first filter only the seasons that we need from defensive data
seasons_needed = df_def['season'].unique()
df_main = df_main[df_main['season'].isin(seasons_needed)]
# print the target seasons
print("Seasons needed for merging:", seasons_needed)

### adding clearance_block_interception col

In [ ]:
#clearances_blocks_interceptions, recoveries, defensive_contribution, tackles are the cols needed to be added to the defensive df

def add_clearances_blocks_interceptions(df):
    df['clearances_blocks_interceptions'] = df['clearances'] + df['blocks'] + df['interceptions']
    return df
df_def = add_clearances_blocks_interceptions(df_def)
# print sample data to verify
print(df_def[['clearances', 'blocks', 'interceptions', 'clearances_blocks_interceptions']].head())

## standarizing the keys for perfect matching

In [ ]:
df_def['join_name'] = df_def['name'].astype(str).str.lower().str.strip()

# 2. Standardize Seasons (Ensure they match "2023-24" format in both)
df_main['join_season'] = df_main['season'].astype(str).str.strip()
df_def['join_season'] = df_def['season'].astype(str).str.strip()

manual_nickname_map = {
    'jorge luiz frello filho': 'jorginho',
    'jonathan castro otto': 'jonny',
    'bruno guimaraes rodriguez moura': 'bruno guimaraes',
    'gabriel teodoro martinelli silva': 'gabriel martinelli',
    'emerson leie de souza junior': 'emerson royal',
    'david raya martin': 'david raya',
    'jose sa': 'jose sa',
    'joao palhinha goncalves alves': 'joao palhinha',
    'thiago alcantara do nascimento': 'thiago',
    'matheus luiz nunes': 'matheus nunes',
    'antony matheus dos santos': 'antony',
    'richarlison de andrade': 'richarlison',
    'bernardo mota veiga de carvalho e silva': 'bernardo silva',
    'ederson santana de moraes': 'ederson',
    'joão filipe iria santos moutinho': 'joão moutinho',
    'fabio henrique tavares': 'fabinho',
    'frederico rodrigues de paula santos': 'fred',
    'lucas tolentino coelho de lima': 'lucas paquetá', # Check accent
    'lucas tolentino coelho de lima': 'lucas paqueta', # Try both if unsure
    'benjamin white': 'ben white',
    'gabriel dos santos magalhães': 'gabriel magalhães', # Now that encoding is fixed, map to short
    'bruno guimarães rodriguez moura': 'bruno guimarães',
    'joão palhinha gonçalves': 'joão palhinha',
    'tomas soucek': 'tomáš souček', # Add accents if defensive has them
    'casemiro': 'casemiro', # If Main has long name, map it. Likely 'carlos henrique casimiro'
    'carlos henrique casimiro': 'casemiro',
    'norberto bercique gomes betuncal': 'beto',
    'jose angel esmoris tasende': 'angelino',
    'juan camilo hernandez suarez': 'cucho',
    'daniel ceballos fernandez': 'dani ceballos',
    'anssumane fati vieira': 'ansu fati',
    'anssumane fati': 'ansu fati',
    'alexandre moreno lopera': 'alex moreno',
    'łukasz fabianski': 'lukasz fabianski', # Fix the Polish 'ł' manually
    'lukasz fabianski': 'lukasz fabianski',  # Safety net
    'ahmed el-sayed hegazy': 'ahmed hegazi',
    'a\x81lex moreno lopera': 'alex moreno',   # Found in your list
    'ivan peria¡ia\x87': 'ivan perisic',       # Found in your list
    'muhamed bea¡ia\x87': 'muhamed besic',     # Found in your list
    'a\x81lex moreno lopera': 'alex moreno',
    'edson a\x81lvarez velazquez': 'edson alvarez',
    
    # The Nicknames & Legal Names
    'abdul fatawu': 'abdul fatawu issahaku',
    'borja gonzalez tomas': 'borja baston',
    'fabio ferreira vieira': 'fabio vieira',
    'fernando luiz rosa': 'fernandinho',
    'giovanni reyna': 'gio reyna',
    'hamed traore': 'hamed junior traore',
    'ian carlo poveda-ocampo': 'ian poveda',
    'jhon duran': 'jader duran',
    'julian araujo zuniga': 'julian araujo',
    'thakgalo leshabela': 'khanya leshabela',
    'francisco casilla cortes': 'kiko casilla',
    'francisco femenia far': 'kiko femenia',
    'marcus oliveira alencar': 'marquinhos',
    'oluwasemilogo adesewo ibidapo ajayi': 'semi ajayi',
    'tariqe fosu-henry': 'tariqe fosu',
    'vini de souza costa': 'vinicius souza',
    'vitor ferreira': 'vitinha',
    'jose reina': 'pepe reina',
    'djordje petrovic': 'đorđe petrovic',  # Matching the defensive spelling
    'jose a\x81ngel esmoris tasende': 'angelino', # Found hidden in candidate list
}

df_main['join_name'] = df_main['join_name'].replace(manual_nickname_map)


# ==========================================
# STEP 3: AUTOMATED SMART MATCHING
# ==========================================
print("--- STARTING SMART MATCHING ---")

# 1. PREPARE LISTS
# We only care about names that are currently missing in the Main DF
# (i.e., names in Main that don't yet match a name in Defensive)
valid_def_names = set(df_def['join_name'].unique())
main_unique = df_main['join_name'].unique()
missing_names = [n for n in main_unique if n not in valid_def_names]

print(f"Attempting to resolve {len(missing_names)} missing names...")

name_mapping = {}

# 2. LOGIC A: SUBSTRING MATCH (The "Gabriel Jesus" Fix)
# We check if a Defensive Name (Short) is fully inside a Main Name (Long)
# e.g. "gabriel jesus" is inside "gabriel fernando de jesus"
for m_name in missing_names:
    m_tokens = set(m_name.split())
    candidates = []
    
    for d_name in valid_def_names:
        d_tokens = set(d_name.split())
        # Check if ALL words in the short name appear in the long name
        if d_tokens.issubset(m_tokens):
            candidates.append(d_name)
    
    if candidates:
        # If multiple matches, pick the longest one (Most specific)
        # Prevents "Gabriel" matching "Gabriel Jesus" incorrectly
        best_match = max(candidates, key=len)
        name_mapping[m_name] = best_match

# 3. LOGIC B: FUZZY MATCH (The "Fabian Schär" Fix)
# For names that didn't match via substring (likely due to spelling/encoding diffs)
# We only check names that Logic A didn't solve
remaining_missing = [n for n in missing_names if n not in name_mapping]
def_name_list = list(valid_def_names)

for m_name in remaining_missing:
    # Cutoff 0.8 is strict to avoid bad matches (we prefer missing data over wrong data)
    matches = difflib.get_close_matches(m_name, def_name_list, n=1, cutoff=0.8)
    if matches:
        name_mapping[m_name] = matches[0]

# 4. APPLY THE UPDATES (To the JOIN KEY only)
print(f"Found {len(name_mapping)} new automatic matches.")
print("Updating 'join_name' column (Original names are safe)...")
df_main['join_name'] = df_main['join_name'].replace(name_mapping)


# ==========================================
# 1. DEFINE ACCENT REMOVER
# ==========================================
def remove_accents(input_str):
    if not isinstance(input_str, str):
        return str(input_str)
    # Normalize unicode characters to decompose them (e.g., 'á' becomes 'a' + '´')
    nfkd_form = unicodedata.normalize('NFKD', input_str)
    # Filter out non-spacing mark characters (the accents)
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)])

print("--- STRIPPING ACCENTS FROM BOTH DATASETS ---")

# Apply to Main
df_main['join_name'] = df_main['join_name'].apply(remove_accents)

# Apply to Defensive
df_def['join_name'] = df_def['join_name'].apply(remove_accents)



### coverage report:

In [ ]:
print("\n=== UNIQUE NAME COVERAGE REPORT ===")

# Get unique names
main_names_set = set(df_main['join_name'].unique())
def_names_set = set(df_def['join_name'].unique())

# Calculate intersection
matched_names = main_names_set.intersection(def_names_set)
missing_def_names = def_names_set - main_names_set

# Metrics
total_def_names = len(def_names_set)
matched_count = len(matched_names)
coverage_pct = (matched_count / total_def_names) * 100

print(f"Unique Defensive Names:   {total_def_names}")
print(f"Found in Main DataFrame:  {matched_count}")
print(f"Defensive Name Coverage:  {coverage_pct:.2f}%")

# now the real merging:

In [ ]:
# ==========================================
# 2. THE MERGE & SMART VALIDATION
# ==========================================
# IMPORTANT: We now use game_number instead of GW for merging
# This handles postponed matches correctly by matching on chronological game order
print("--- MERGE DIAGNOSTICS (Using game_number) ---")

# 1. PREPARE DEFENSIVE DATA
cols_cbi = ['clearances', 'blocks', 'interceptions']
df_def[cols_cbi] = df_def[cols_cbi].fillna(0)

if 'clearances_blocks_interceptions' not in df_def.columns:
    df_def['clearances_blocks_interceptions'] = (
        df_def['clearances'] + df_def['blocks'] + df_def['interceptions']
    )

# Select Merge Subset - NOW USING game_number INSTEAD OF GW
def_subset = df_def[[
    'join_name', 'game_number', 'join_season', 
    'tackles', 'clearances_blocks_interceptions'
]].rename(columns={
    'tackles': 'tackles_new', 
    'clearances_blocks_interceptions': 'cbi_new'
})

# ---------------------------------------------------------
# SMART METRIC: "Can we match it?"
# ---------------------------------------------------------
# Using game_number for matching ensures correct alignment even with postponed matches
main_keys = set(zip(df_main['join_name'], df_main['game_number'], df_main['join_season']))
def_keys = set(zip(def_subset['join_name'], def_subset['game_number'], def_subset['join_season']))

# The Intersection: These are the rows that SHOULD merge successfully
possible_matches = main_keys.intersection(def_keys)
print(f"Total Rows in Main: {len(df_main)}")
print(f"Rows with available Defensive Data: {len(possible_matches)}")

# 2. PERFORM LEFT MERGE - NOW ON game_number
merged_df = pd.merge(
    df_main, 
    def_subset, 
    on=['join_name', 'game_number', 'join_season'], 
    how='left'
)

# ---------------------------------------------------------
# REAL VALIDATION: DID THE MERGE WORK?
# ---------------------------------------------------------
merged_df['key_tuple'] = list(zip(merged_df['join_name'], merged_df['game_number'], merged_df['join_season']))
should_have_data = merged_df[merged_df['key_tuple'].isin(possible_matches)]

# Check if they are actually filled
successful_merges = should_have_data['tackles_new'].notna().sum()
technical_success_rate = (successful_merges / len(should_have_data)) * 100 if len(should_have_data) > 0 else 0

print(f"\nTechnical Merge Success Rate: {technical_success_rate:.2f}%")
print("(This should be 100%. It means every row that existed in the source was successfully merged.)")

# ==========================================
# 3. UPDATE STATS & FINAL REPORT
# ==========================================
# Update Tackles
merged_df['tackles'] = np.where(
    merged_df['tackles_new'].notna(), 
    merged_df['tackles_new'], 
    np.where(merged_df['minutes'] == 0, 0, merged_df['tackles'].fillna(0))
)

# Update CBI
old_cbi = merged_df['clearances_blocks_interceptions'] if 'clearances_blocks_interceptions' in merged_df.columns else 0
merged_df['clearances_blocks_interceptions'] = np.where(
    merged_df['cbi_new'].notna(), 
    merged_df['cbi_new'], 
    np.where(merged_df['minutes'] == 0, 0, old_cbi)
)

# Clean up temps - keep game_number as it's useful for feature engineering
merged_df.drop(columns=['tackles_new', 'cbi_new', 'join_name', 'join_season', 'is_matched', 'key_tuple'], inplace=True, errors='ignore')

print("\n✓ Merge complete using game_number for correct chronological alignment")

# here validate that after merging we made the results in all seasons data

In [ ]:
# ==========================================
# STEP 4: COMBINE MERGED DATA BACK INTO FULL DATASET
# ==========================================
# Problem: We filtered df_main to only seasons with defensive data (2019-20 to 2024-25)
# Solution: Reload the full dataset and combine:
#   - Seasons 2016-17 to 2018-19: Already have real defensive stats
#   - Seasons 2019-20 to 2024-25: Use merged_df with newly merged defensive stats
#   - Season 2025-26: Keep as-is (if not in defensive data)

print("="*80)
print("STEP 4: Combining merged data back into full dataset")
print("="*80)

# 1. Reload the full all_seasons_data (before we filtered it)
df_full = pd.read_csv("all_seasons_data.csv")
print(f"Full dataset shape: {df_full.shape}")
print(f"Seasons in full dataset: {sorted(df_full['season'].unique())}")

# 2. Get the seasons that were merged with defensive data
merged_seasons = merged_df['season'].unique().tolist()
print(f"\nSeasons that were merged with defensive stats: {merged_seasons}")

# 3. Get the seasons that were NOT merged (already have defensive data or no data available)
seasons_not_merged = [s for s in df_full['season'].unique() if s not in merged_seasons]
print(f"Seasons NOT merged (already have defensive data): {seasons_not_merged}")

# 4. Extract the non-merged seasons from the full dataset
df_not_merged = df_full[df_full['season'].isin(seasons_not_merged)].copy()
print(f"Records from non-merged seasons: {len(df_not_merged):,}")

# 5. Ensure both DataFrames have the same columns
# Add game_number to non-merged seasons if missing (set to None for older seasons)
if 'game_number' not in df_not_merged.columns:
    df_not_merged['game_number'] = None
    
# Align columns - get the union of both column sets
all_columns = list(set(merged_df.columns) | set(df_not_merged.columns))
for col in all_columns:
    if col not in merged_df.columns:
        merged_df[col] = None
    if col not in df_not_merged.columns:
        df_not_merged[col] = None

# 6. Combine merged seasons with non-merged seasons
all_seasons_final = pd.concat([df_not_merged, merged_df], ignore_index=True)

# 7. Sort by season and element for consistency
all_seasons_final = all_seasons_final.sort_values(['season', 'element', 'GW']).reset_index(drop=True)

print(f"\n✅ Final combined dataset shape: {all_seasons_final.shape}")
print(f"Seasons in final dataset: {sorted(all_seasons_final['season'].unique())}")

# 8. Verify record counts by season
print("\nRecords per season:")
print(all_seasons_final.groupby('season').size().sort_index())

## Step 5: Recalculate defensive_contribution and modify points for merged seasons

In [ ]:
# ==========================================
# STEP 5: RECALCULATE DEFENSIVE CONTRIBUTION & MODIFY POINTS
# ==========================================
# Now that we have the real defensive stats for 2019-20 to 2024-25,
# we need to recalculate defensive_contribution and apply the point modifications

print("="*80)
print("STEP 5: Recalculating defensive contribution for merged seasons")
print("="*80)

# 1. Define the defensive contribution calculation function
def calculate_defensive_contribution_row(row):
    """Calculate defensive contribution based on position"""
    position = row['position']
    cbi = row.get('clearances_blocks_interceptions', 0) or 0
    tackles = row.get('tackles', 0) or 0
    recoveries = row.get('recoveries', 0) or 0
    
    if position == 'DEF':
        return cbi + tackles
    elif position in ['MID', 'FWD']:
        return cbi + tackles + recoveries
    else:  # GK or unknown
        return 0

# 2. Recalculate defensive_contribution for seasons that were merged
# (2019-20 to 2024-25 now have real defensive stats)
merged_season_mask = all_seasons_final['season'].isin(merged_seasons)

print(f"Recalculating defensive_contribution for {merged_season_mask.sum():,} records...")

all_seasons_final.loc[merged_season_mask, 'defensive_contribution'] = \
    all_seasons_final.loc[merged_season_mask].apply(calculate_defensive_contribution_row, axis=1)

# 3. Apply point modification to merged seasons (as per FPL rules)
# Add 2 bonus points when:
# - Defenders reach defensive_contribution >= 10
# - Midfielders/Forwards reach defensive_contribution >= 12

def calculate_modified_points(row):
    """Add 2 bonus points for defensive contribution threshold"""
    points = row['total_points']
    position = row['position']
    def_contrib = row.get('defensive_contribution', 0) or 0
    
    if position == 'DEF' and def_contrib >= 10:
        points += 2
    elif position in ['MID', 'FWD'] and def_contrib >= 12:
        points += 2
    return points

# Only modify points for the merged seasons (2019-20 to 2024-25)
# Seasons 2016-17 to 2018-19 already had points modified earlier
print("Applying point modifications for defensive contributions...")

all_seasons_final.loc[merged_season_mask, 'total_points'] = \
    all_seasons_final.loc[merged_season_mask].apply(calculate_modified_points, axis=1)

print("✅ Defensive contribution recalculated and points modified")

# 4. Verify the results
print("\nSample of defensive stats after recalculation:")
sample_cols = ['name', 'season', 'position', 'tackles', 'clearances_blocks_interceptions', 
               'defensive_contribution', 'total_points']
available_cols = [c for c in sample_cols if c in all_seasons_final.columns]
print(all_seasons_final[all_seasons_final['season'] == '2023-24'][available_cols].head(10))

## Step 6: Save the final complete dataset

In [ ]:
# ==========================================
# STEP 6: SAVE THE FINAL COMPLETE DATASET
# ==========================================
print("="*80)
print("STEP 6: Saving final dataset")
print("="*80)

# Save the complete dataset with all seasons and merged defensive stats
output_path = 'all_seasons_data_final.csv'
all_seasons_final.to_csv(output_path, index=False, encoding='utf-8')

print(f"✅ Saved: {output_path}")
print(f"   Total records: {len(all_seasons_final):,}")
print(f"   Total columns: {len(all_seasons_final.columns)}")
print(f"   Seasons: {sorted(all_seasons_final['season'].unique())}")

# Summary statistics
print("\n" + "="*80)
print("FINAL DATASET SUMMARY")
print("="*80)
print(f"\nRecords by season:")
print(all_seasons_final.groupby('season').size().sort_index())

print(f"\nDefensive stats coverage (non-zero defensive_contribution):")
def_contrib_stats = all_seasons_final.groupby('season').apply(
    lambda x: (x['defensive_contribution'] > 0).sum() / len(x) * 100
)
print(def_contrib_stats.round(1).to_string())

print("\n" + "="*80)
print("DATA PREPARATION COMPLETE - READY FOR FEATURE ENGINEERING")
print("="*80)

# now we make the previous game and that stuff:

### load the dataset

In [ ]:
all_seasons_data = pd.read_csv('all_seasons_data_final.csv')

## defining the function to add the rollback stats

In [ ]:
def add_previous_game_stats(df, n_gameweeks=5, use_cross_season=False):
    """
    Add columns for each stat for the previous N games to each player.
    
    IMPORTANT: Uses 'game_number' instead of 'GW' for correct chronological ordering.
    This handles postponed matches where a GW5 match might be played during GW20.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing player statistics with columns including 'name', 'element', 'game_number', 'season'
    n_games : int, default=5
        Number of previous games to include
    use_cross_season : bool, default=False
        If True, carry over stats from previous season
        If False, stats only come from within the same season (Game 1 will have NaN for previous stats)
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with additional columns for previous game statistics
    """
    # Columns to exclude from previous gameweek calculation
    exclude_columns = ['name', 'element', 'GW', 'game_number', 'position', 'team', 
                       'season', 'fixture', 'kickoff_time', 'round', ]
    
    # Get stat columns (only numeric columns except the excluded ones)
    stat_columns = [col for col in df.columns 
                    if col not in exclude_columns and pd.api.types.is_numeric_dtype(df[col])]
    
    # Sort by player (element), season, and GAME_NUMBER for correct chronological order
    # This ensures postponed matches are in the right sequence
    df_sorted = df.sort_values(['element', 'season', 'game_number']).reset_index(drop=True)
    
    # Create a copy to avoid modifying the original
    df_result = df_sorted.copy()
    
    # Determine groupby columns based on cross_season parameter
    if use_cross_season:
        # Group only by element - allows stats to carry across seasons
        group_cols = ['element']
    else:
        # Group by element and season - stats only within same season
        group_cols = ['element', 'season']
    
    # For each stat column, create N previous game columns
    for stat in stat_columns:
        for i in range(1, n_gameweeks + 1):
            col_name = f'{stat}_prev_{i}'
            # Shift by i positions for each player (based on game_number order)
            df_result[col_name] = df_result.groupby(group_cols, sort=False)[stat].shift(i)
    
    return df_result

### apllying the function

In [ ]:
# Load data if needed
# all_seasons_data = pd.read_csv('all_seasons_data.csv', index_col=0)

# Apply the function to create lagged features
# Parameters:
#   n_gameweeks: Number of previous gameweeks to include (default=5)
#   use_cross_season: Whether to carry stats across seasons (default=False)
all_seasons_data_with_prev = add_previous_game_stats(all_seasons_data, n_gameweeks=5, use_cross_season=False)

# Check the result
print("Original shape:", all_seasons_data.shape)
print("New shape:", all_seasons_data_with_prev.shape)
print("\nSample of new columns:")
print(all_seasons_data_with_prev.filter(regex='_prev_').columns.tolist()[:20])

## Feature Engineering: Opponent Strength
Calculate opponent strength metrics based on team performance to capture fixture difficulty.

In [ ]:
def add_opponent_strength_features(df, rolling_window=5):
    """
    Add opponent strength features based on historical team performance.
    
    IMPORTANT: Uses 'game_number' instead of 'GW' for correct chronological ordering.
    This handles postponed matches where teams don't play in certain gameweeks.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Player-level data with team and opponent_team columns
    rolling_window : int, default=5
        Number of previous games to calculate rolling team strength
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with added opponent strength features
    """
    
    # Sort by season and game_number for correct chronological order
    df = df.sort_values(['season', 'game_number', 'team']).reset_index(drop=True)
    
    # Step 1: Calculate team-level aggregates per game_number
    # We use game_number to ensure correct ordering even with postponed matches
    team_stats = df.groupby(['season', 'game_number', 'team']).agg({
        'goals_scored': 'sum',           # Total goals scored by team
        'goals_conceded': 'sum',         # Total goals conceded
        'clean_sheets': 'max',           # 1 if clean sheet, 0 otherwise
        'total_points': 'sum',           # Total team points
        'assists': 'sum',
        'saves': 'sum',
        'bonus': 'sum'
    }).reset_index()
    
    # Rename for clarity
    team_stats.columns = ['season', 'game_number', 'team', 'team_goals_scored', 
                          'team_goals_conceded', 'team_clean_sheet', 
                          'team_total_points', 'team_assists', 
                          'team_saves', 'team_bonus']
    
    # Step 2: Calculate rolling averages for team strength
    # Sort by team, season, and game_number for correct rolling calculation
    team_stats = team_stats.sort_values(['team', 'season', 'game_number'])
    
    for col in ['team_goals_scored', 'team_goals_conceded', 'team_clean_sheet', 'team_total_points']:
        # Rolling average (last N games)
        team_stats[f'{col}_rolling_{rolling_window}'] = (
            team_stats.groupby(['team', 'season'])[col]
            .rolling(window=rolling_window, min_periods=1)
            .mean()
            .reset_index(level=[0, 1], drop=True)
        )
    
    # Step 3: Calculate defensive and offensive strength ratings
    # Defensive strength = inverse of goals conceded (lower is better)
    team_stats['defensive_strength'] = (
        1 / (team_stats[f'team_goals_conceded_rolling_{rolling_window}'] + 1)
    )
    
    # Offensive strength = goals scored
    team_stats['offensive_strength'] = (
        team_stats[f'team_goals_scored_rolling_{rolling_window}']
    )
    
    # Overall team strength (normalized 0-100 scale)
    team_stats['overall_team_strength'] = (
        team_stats[f'team_total_points_rolling_{rolling_window}']
    )
    
    # Step 4: Shift team stats forward by 1 game (so we use PAST performance to predict CURRENT game)
    for col in team_stats.columns:
        if col not in ['season', 'game_number', 'team']:
            team_stats[col] = team_stats.groupby(['team', 'season'])[col].shift(1)
    
    # Increment game_number for joining (we want opponent stats from their previous games)
    team_stats['game_number'] = team_stats['game_number'] + 1
    
    # Step 5: Join opponent strength to main dataframe
    # Prepare opponent stats with 'opponent_' prefix
    opponent_stats = team_stats.copy()
    opponent_stats.columns = ['season', 'game_number', 'opponent_team'] + [
        'opponent_' + col for col in opponent_stats.columns if col not in ['season', 'game_number', 'team']
    ]
    
    # Merge opponent stats using game_number
    df = df.merge(
        opponent_stats[['season', 'game_number', 'opponent_team', 
                       'opponent_defensive_strength', 
                       'opponent_offensive_strength',
                       'opponent_overall_team_strength']],
        on=['season', 'game_number', 'opponent_team'],
        how='left'
    )
    
    # Step 6: Add relative strength features
    # First, add own team strength
    df = df.merge(
        team_stats[['season', 'game_number', 'team', 
                   'defensive_strength', 
                   'offensive_strength',
                   'overall_team_strength']],
        on=['season', 'game_number', 'team'],
        how='left',
        suffixes=('', '_own')
    )
    
    # Calculate relative strength (advantage)
    df['offensive_advantage'] = df['offensive_strength'] - df['opponent_defensive_strength']
    df['defensive_advantage'] = df['defensive_strength'] - df['opponent_offensive_strength']
    df['overall_advantage'] = df['overall_team_strength'] - df['opponent_overall_team_strength']
    
    # Step 7: Add difficulty rating (normalized)
    # Higher value = more difficult opponent
    df['opponent_difficulty'] = (
        df['opponent_overall_team_strength'] / df['overall_team_strength']
    ).fillna(1)
    
    return df

### Apply Opponent Strength Features
Add team and opponent strength metrics to the dataset.

In [ ]:
# Apply opponent strength features to the dataset
all_seasons_data_with_opponent = add_opponent_strength_features(
    all_seasons_data_with_prev, 
    rolling_window=5
)

# Check new features
print("Shape after adding opponent features:", all_seasons_data_with_opponent.shape)
print("\nNew opponent strength columns:")
opponent_cols = [col for col in all_seasons_data_with_opponent.columns if 'opponent' in col.lower() or 'advantage' in col.lower() or 'difficulty' in col.lower()]
print(opponent_cols)

## Feature Engineering: Rolling Averages (Player Form)
Calculate moving averages to capture short-term and long-term player performance trends.

In [ ]:
def add_rolling_player_stats(df, windows=[3, 5, 10]):
    """
    Add rolling averages for player performance metrics to capture form.
    
    IMPORTANT: Uses 'game_number' instead of 'GW' for correct chronological ordering.
    This ensures rolling averages are calculated based on actual game sequence,
    not gameweek numbers (which can be out of order due to postponements).
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Player-level data
    windows : list, default=[3, 5, 10]
        List of rolling window sizes (3-game, 5-game, 10-game form)
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with added rolling average features
    """
    
    # Key performance metrics to calculate rolling averages for
    rolling_stats = ['total_points', 'goals_scored', 'assists', 'minutes', 
                     'bonus', 'bps', 'clean_sheets', 'saves', 
                     'ict_index', 'creativity', 'threat', 'influence']
    
    # Sort by player and game_number for correct chronological order
    df = df.sort_values(['element', 'season', 'game_number']).reset_index(drop=True)
    
    # Calculate rolling averages for each window size
    for window in windows:
        for stat in rolling_stats:
            if stat in df.columns:
                col_name = f'{stat}_rolling_{window}'
                # Calculate rolling mean based on game_number order, shift by 1 to avoid data leakage
                df[col_name] = (
                    df.groupby(['element', 'season'])[stat]
                    .rolling(window=window, min_periods=1)
                    .mean()
                    .reset_index(level=[0, 1], drop=True)
                    .shift(1)  # Shift to use only past data
                )
    
    return df

## Feature Engineering: Additional Context Features
Add home/away indicators, season progression, and price momentum features.

In [ ]:
def add_context_features(df):
    """
    Add contextual features like home/away, season progression, and price changes.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Player-level data
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with added context features
    """
    
    # 1. Home/Away indicator
    df['is_home'] = df['was_home'].astype(int)
    
    # 2. Season progression features
    # Season stage: early (GW 1-13), mid (GW 14-26), late (GW 27-38)
    df['season_stage'] = pd.cut(df['GW'], bins=[0, 13, 26, 38], 
                                 labels=['early', 'mid', 'late'])
    
    # Gameweek as percentage of season completion
    df['season_progress'] = df['GW'] / 38.0
    
    # 3. Price change features (if value column exists)
    if 'value' in df.columns:
        # Sort by player and time
        df = df.sort_values(['element', 'season', 'GW']).reset_index(drop=True)
        
        # Price change from previous gameweek
        df['price_change'] = df.groupby(['element', 'season'])['value'].diff()
        
        # Cumulative price change within season (from starting price)
        df['price_change_cumulative'] = df.groupby(['element', 'season'])['value'].transform(
            lambda x: x - x.iloc[0] if len(x) > 0 else 0
        )
        
        # Price trend: increasing (1), stable (0), decreasing (-1)
        df['price_trend'] = df['price_change'].apply(
            lambda x: 1 if x > 0 else (-1 if x < 0 else 0)
        )
    
    # 4. Match number within season (alternative to GW)
    df['match_number'] = df.groupby(['element', 'season']).cumcount() + 1
    
    return df

### Apply All Feature Engineering Functions
Combine all feature engineering steps to create the complete dataset.

In [ ]:
# Apply rolling averages for player form
print("Adding rolling averages...")
all_seasons_data_with_rolling = add_rolling_player_stats(
    all_seasons_data_with_opponent, 
    windows=[3, 5, 10]
)

# Apply context features
print("Adding context features...")
all_seasons_data_featured = add_context_features(all_seasons_data_with_rolling)

# Check final shape
print(f"\nFinal dataset shape: {all_seasons_data_featured.shape}")
print(f"Original dataset shape: {all_seasons_data.shape}")
print(f"Total new features added: {all_seasons_data_featured.shape[1] - all_seasons_data.shape[1]}")

In [ ]:
## save the final dataset with features
all_seasons_data_featured.to_csv('all_seasons_data_featured.csv', index=False)

# consider here making some cleaning

# here I am assuming that the work with data is done we start standarizing and training

# Position-Specific Model Training

This section implements position-specific models for more accurate FPL predictions:
- **Section 2**: Fixture Difficulty Rating (FDR) features
- **Section 3**: Position-specific feature engineering
- **Section 4**: Hyperparameter tuning with GridSearchCV
- **Section 5**: Ensemble methods (Stacking and Blending)
- **Section 6**: Training separate models for GK, DEF, MID, FWD

## Section 2: Load Fixture Difficulty Data

Add FDR (Fixture Difficulty Rating) features to capture match difficulty.

In [ ]:
# Load the dataset with all features
df_main = pd.read_csv('all_seasons_data_featured.csv')

print(f"Dataset shape: {df_main.shape}")
print(f"Seasons available: {sorted(df_main['season'].unique())}")
print(f"\nColumns available: {df_main.columns.tolist()[:20]}...")  # Show first 20 columns

In [ ]:
# Import additional libraries needed for position-specific models
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV, 
    cross_val_score, TimeSeriesSplit
)
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor, 
    StackingRegressor, VotingRegressor, AdaBoostRegressor
)
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
import joblib

warnings.filterwarnings('ignore')

## Section 3: Feature Engineering for Position-Specific Models

Define position-specific feature sets that capture the unique characteristics of each position.

In [ ]:
# Standardize position names to match advanced_fpl_models format
position_mapping = {
    'Goalkeeper': 'GK',
    'Defender': 'DEF',
    'Midfielder': 'MID',
    'Forward': 'FWD'
}

df_main['position_short'] = df_main['position'].map(position_mapping)

print("Position distribution:")
print(df_main['position_short'].value_counts())

# Define position-specific feature sets
COMMON_FEATURES = [
    'value', 'is_home', 'season_progress', 'match_number',
    'total_points_rolling_3', 'total_points_rolling_5', 'total_points_rolling_10',
    'minutes_rolling_3', 'minutes_rolling_5', 'minutes_rolling_10',
    'bps_rolling_3', 'bps_rolling_5', 'bps_rolling_10',
    'ict_index_rolling_3', 'ict_index_rolling_5', 'ict_index_rolling_10'
]

GK_FEATURES = COMMON_FEATURES + [
    'saves_rolling_3', 'saves_rolling_5', 'saves_rolling_10',
    'clean_sheets_rolling_3', 'clean_sheets_rolling_5', 'clean_sheets_rolling_10',
    'opponent_strength_attack', 'opponent_form'
]

DEF_FEATURES = COMMON_FEATURES + [
    'clean_sheets_rolling_3', 'clean_sheets_rolling_5', 'clean_sheets_rolling_10',
    'goals_scored_rolling_3', 'assists_rolling_3',
    'tackles_rolling_3', 'tackles_rolling_5',
    'threat_rolling_3', 'threat_rolling_5',
    'opponent_strength_attack', 'opponent_form', 'defensive_advantage'
]

MID_FEATURES = COMMON_FEATURES + [
    'goals_scored_rolling_3', 'goals_scored_rolling_5', 'goals_scored_rolling_10',
    'assists_rolling_3', 'assists_rolling_5', 'assists_rolling_10',
    'creativity_rolling_3', 'creativity_rolling_5', 'creativity_rolling_10',
    'threat_rolling_3', 'threat_rolling_5', 'threat_rolling_10',
    'influence_rolling_3', 'influence_rolling_5', 'influence_rolling_10',
    'tackles_rolling_3',
    'opponent_strength_defense', 'opponent_form', 'attacking_advantage'
]

FWD_FEATURES = COMMON_FEATURES + [
    'goals_scored_rolling_3', 'goals_scored_rolling_5', 'goals_scored_rolling_10',
    'assists_rolling_3', 'assists_rolling_5', 'assists_rolling_10',
    'threat_rolling_3', 'threat_rolling_5', 'threat_rolling_10',
    'creativity_rolling_3', 'creativity_rolling_5',
    'influence_rolling_3', 'influence_rolling_5',
    'opponent_strength_defense', 'opponent_form', 'attacking_advantage'
]

POSITION_FEATURES = {
    'GK': GK_FEATURES,
    'DEF': DEF_FEATURES,
    'MID': MID_FEATURES,
    'FWD': FWD_FEATURES
}

print("\nPosition-specific feature sets defined!")
for pos, features in POSITION_FEATURES.items():
    print(f"{pos}: {len(features)} features")

In [ ]:
# Prepare data for modeling
def prepare_position_data(df, position, features):
    """Prepare data for a specific position"""
    
    # Filter by position
    pos_df = df[df['position_short'] == position].copy()
    
    # Get available features (some may not exist)
    available_features = [f for f in features if f in pos_df.columns]
    missing_features = [f for f in features if f not in pos_df.columns]
    
    if missing_features:
        print(f"  Missing features for {position}: {missing_features}")
    
    # Remove rows with missing target
    pos_df = pos_df.dropna(subset=['total_points'])
    
    # Fill missing features with 0
    for col in available_features:
        pos_df[col] = pos_df[col].fillna(0)
    
    # Remove infinite values
    pos_df = pos_df.replace([np.inf, -np.inf], 0)
    
    X = pos_df[available_features]
    y = pos_df['total_points']
    
    return X, y, available_features, pos_df

print("Data preparation function defined!")

## Section 4: Hyperparameter Tuning with GridSearchCV

Define hyperparameter grids and tuning functions for optimal model performance.

In [ ]:
# Define hyperparameter grids for different models
PARAM_GRIDS = {
    'RandomForest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 15, 20, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2', None]
    },
    'GradientBoosting': {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'max_depth': [3, 5, 7, 9],
        'min_samples_split': [2, 5, 10],
        'subsample': [0.8, 0.9, 1.0]
    },
    'Ridge': {
        'alpha': [0.01, 0.1, 1, 10, 100]
    },
    'ElasticNet': {
        'alpha': [0.01, 0.1, 1],
        'l1_ratio': [0.2, 0.5, 0.8]
    }
}

PARAM_GRIDS['XGBoost'] = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0]
}

PARAM_GRIDS['LightGBM'] = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7, -1],
    'num_leaves': [31, 50, 100],
    'subsample': [0.8, 0.9, 1.0]
}

print("Hyperparameter grids defined for models:")
for model_name in PARAM_GRIDS:
    print(f"  - {model_name}")

In [ ]:
def tune_model(model, param_grid, X_train, y_train, cv=5):
    """Perform GridSearchCV for hyperparameter tuning"""
    
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=cv,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    return grid_search.best_estimator_, grid_search.best_params_, grid_search.best_score_

def tune_model_randomized(model, param_distributions, X_train, y_train, n_iter=50, cv=5):
    """Perform RandomizedSearchCV for faster hyperparameter tuning"""
    
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions,
        n_iter=n_iter,
        cv=cv,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1,
        random_state=42
    )
    
    random_search.fit(X_train, y_train)
    
    return random_search.best_estimator_, random_search.best_params_, random_search.best_score_

print("Hyperparameter tuning functions defined!")

## Section 5: Ensemble Methods - Stacking and Blending

Create advanced ensemble models that combine multiple base models for better predictions.

In [ ]:
def create_stacking_model(base_models=None, meta_model=None):
    """Create a stacking ensemble model"""
    
    if base_models is None:
        base_models = [
            ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingRegressor(n_estimators=100, random_state=42)),
            ('ridge', Ridge(alpha=1.0))
        ]

        base_models.append(('xgb', xgb.XGBRegressor(n_estimators=100, random_state=42, verbosity=0)))
        base_models.append(('lgbm', lgb.LGBMRegressor(n_estimators=100, random_state=42, verbose=-1)))
    
    if meta_model is None:
        meta_model = Ridge(alpha=1.0)
    
    stacking_model = StackingRegressor(
        estimators=base_models,
        final_estimator=meta_model,
        cv=5,
        n_jobs=-1
    )
    
    return stacking_model

def create_voting_model(models=None, weights=None):
    """Create a voting ensemble model (averaging)"""
    
    if models is None:
        models = [
            ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
            ('gb', GradientBoostingRegressor(n_estimators=100, random_state=42)),
            ('ridge', Ridge(alpha=1.0))
        ]
    
    voting_model = VotingRegressor(
        estimators=models,
        weights=weights,
        n_jobs=-1
    )
    
    return voting_model

print("Ensemble model functions defined!")

In [ ]:
class BlendingEnsemble:
    """Custom blending ensemble that trains models on different data splits"""
    
    def __init__(self, base_models, meta_model, blend_ratio=0.5):
        self.base_models = base_models
        self.meta_model = meta_model
        self.blend_ratio = blend_ratio
        self.fitted_base_models = []
        
    def fit(self, X, y):
        # Split data for blending
        n_blend = int(len(X) * self.blend_ratio)
        X_train, X_blend = X.iloc[:n_blend], X.iloc[n_blend:]
        y_train, y_blend = y.iloc[:n_blend], y.iloc[n_blend:]
        
        # Train base models on first portion
        blend_predictions = np.zeros((len(X_blend), len(self.base_models)))
        
        for i, (name, model) in enumerate(self.base_models):
            model.fit(X_train, y_train)
            self.fitted_base_models.append(model)
            blend_predictions[:, i] = model.predict(X_blend)
        
        # Train meta model on blend predictions
        self.meta_model.fit(blend_predictions, y_blend)
        
        # Retrain base models on full data
        for model in self.fitted_base_models:
            model.fit(X, y)
        
        return self
    
    def predict(self, X):
        # Get predictions from all base models
        predictions = np.zeros((len(X), len(self.fitted_base_models)))
        
        for i, model in enumerate(self.fitted_base_models):
            predictions[:, i] = model.predict(X)
        
        # Use meta model to combine predictions
        return self.meta_model.predict(predictions)

print("BlendingEnsemble class defined!")

## Section 6: Position-Specific Model Training

Train separate models for each position (GK, DEF, MID, FWD) to capture position-specific patterns.

**Training time**: ~2-5 minutes depending on your system.

In [ ]:
class PositionSpecificModels:
    """Train and manage separate models for each position"""
    
    def __init__(self):
        self.models = {}
        self.scalers = {}
        self.feature_sets = POSITION_FEATURES
        self.actual_features = {}  # Store actual features used during training
        self.performance_metrics = {}
        
    def train_all_positions(self, df, use_ensemble=True, tune_hyperparameters=False):
        """Train models for all positions"""
        
        positions = ['GK', 'DEF', 'MID', 'FWD']
        
        for position in positions:
            print(f"\n{'='*50}")
            print(f"Training model for {position}")
            print('='*50)
            
            # Prepare data
            features = self.feature_sets.get(position, COMMON_FEATURES)
            X, y, available_features, pos_df = prepare_position_data(df, position, features)
            
            if len(X) < 100:
                print(f"Insufficient data for {position}: {len(X)} samples")
                continue
            
            print(f"Data shape: {X.shape}")
            print(f"Features used: {len(available_features)}")
            
            # Split data
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42
            )
            
            # Scale features
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            self.scalers[position] = scaler
            self.actual_features[position] = available_features  # Store features used
            
            if use_ensemble:
                # Create stacking ensemble
                model = create_stacking_model()
            else:
                model = GradientBoostingRegressor(n_estimators=200, random_state=42)
            
            if tune_hyperparameters and not use_ensemble:
                print("Tuning hyperparameters...")
                model, best_params, _ = tune_model_randomized(
                    model, PARAM_GRIDS['GradientBoosting'],
                    X_train_scaled, y_train, n_iter=20
                )
                print(f"Best params: {best_params}")
            else:
                model.fit(X_train_scaled, y_train)
            
            self.models[position] = model
            
            # Evaluate
            y_pred = model.predict(X_test_scaled)
            
            mse = mean_squared_error(y_test, y_pred)
            mae = mean_absolute_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)
            
            self.performance_metrics[position] = {
                'mse': mse, 'rmse': np.sqrt(mse), 'mae': mae, 'r2': r2
            }
            
            print(f"\nPerformance for {position}:")
            print(f"  RMSE: {np.sqrt(mse):.3f}")
            print(f"  MAE: {mae:.3f}")
            print(f"  R²: {r2:.3f}")
        
        return self
    
    def predict(self, df, position):
        """Make predictions for a specific position"""
        if position not in self.models:
            raise ValueError(f"No model trained for position: {position}")
        
        # Use the EXACT features that were used during training
        features = self.actual_features.get(position, COMMON_FEATURES)
        X, _, available_features, _ = prepare_position_data(df, position, features)
        
        X_scaled = self.scalers[position].transform(X)
        predictions = self.models[position].predict(X_scaled)
        
        return predictions
    
    def save_models(self, path='models/'):
        """Save all trained models"""
        import os
        os.makedirs(path, exist_ok=True)
        
        for position in self.models:
            joblib.dump(self.models[position], f"{path}{position}_model.pkl")
            joblib.dump(self.scalers[position], f"{path}{position}_scaler.pkl")
            joblib.dump(self.actual_features[position], f"{path}{position}_features.pkl")
        
        print(f"Models saved to {path}")
    
    def load_models(self, path='models/'):
        """Load saved models"""
        for position in ['GK', 'DEF', 'MID', 'FWD']:
            try:
                self.models[position] = joblib.load(f"{path}{position}_model.pkl")
                self.scalers[position] = joblib.load(f"{path}{position}_scaler.pkl")
                self.actual_features[position] = joblib.load(f"{path}{position}_features.pkl")
            except FileNotFoundError:
                print(f"Model for {position} not found")
        
        print("Models loaded!")

print("PositionSpecificModels class defined!")

In [ ]:
# Train position-specific models
print("Training position-specific models...")
print("This may take several minutes...\n")

position_models = PositionSpecificModels()
position_models.train_all_positions(df_main, use_ensemble=True, tune_hyperparameters=False)

In [ ]:
# Display performance summary
print("\n" + "="*60)
print("POSITION-SPECIFIC MODEL PERFORMANCE SUMMARY")
print("="*60)

performance_df = pd.DataFrame(position_models.performance_metrics).T
performance_df = performance_df.round(3)
print(performance_df)

# Visualize model performance
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: RMSE by position
axes[0].bar(performance_df.index, performance_df['rmse'], color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[0].set_title('RMSE by Position', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Position')
axes[0].set_ylabel('RMSE')
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: MAE by position
axes[1].bar(performance_df.index, performance_df['mae'], color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[1].set_title('MAE by Position', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Position')
axes[1].set_ylabel('MAE')
axes[1].grid(axis='y', alpha=0.3)

# Plot 3: R² by position
axes[2].bar(performance_df.index, performance_df['r2'], color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[2].set_title('R² Score by Position', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Position')
axes[2].set_ylabel('R² Score')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Save the trained models
position_models.save_models(path='models/')

print("\n✅ Position-specific models training complete!")
print("   Models saved to 'models/' directory")
print("   You can now use these models for predictions on new data")

## Model Comparison: Position-Specific vs Single Model

Compare the performance of position-specific models against a single unified model.

In [ ]:
# Train a single unified model for comparison
print("Training unified model for comparison...")

# Prepare unified data
unified_features = ['value', 'is_home', 'season_progress', 'match_number',
                   'total_points_rolling_3', 'total_points_rolling_5',
                   'minutes_rolling_3', 'minutes_rolling_5',
                   'bps_rolling_3', 'bps_rolling_5',
                   'ict_index_rolling_3', 'ict_index_rolling_5']

# Filter data with valid target
df_unified = df_main.dropna(subset=['total_points'])

# Get available features
available_unified = [f for f in unified_features if f in df_unified.columns]
X_unified = df_unified[available_unified].fillna(0).replace([np.inf, -np.inf], 0)
y_unified = df_unified['total_points']

# Split and scale
X_train_uni, X_test_uni, y_train_uni, y_test_uni = train_test_split(
    X_unified, y_unified, test_size=0.2, random_state=42
)

scaler_unified = StandardScaler()
X_train_uni_scaled = scaler_unified.fit_transform(X_train_uni)
X_test_uni_scaled = scaler_unified.transform(X_test_uni)

# Train unified model
unified_model = create_stacking_model()
unified_model.fit(X_train_uni_scaled, y_train_uni)

# Evaluate
y_pred_uni = unified_model.predict(X_test_uni_scaled)
mse_uni = mean_squared_error(y_test_uni, y_pred_uni)
mae_uni = mean_absolute_error(y_test_uni, y_pred_uni)
r2_uni = r2_score(y_test_uni, y_pred_uni)

print(f"\nUnified Model Performance:")
print(f"  RMSE: {np.sqrt(mse_uni):.3f}")
print(f"  MAE: {mae_uni:.3f}")
print(f"  R²: {r2_uni:.3f}")

# Compare average performance
avg_rmse_pos = performance_df['rmse'].mean()
avg_mae_pos = performance_df['mae'].mean()
avg_r2_pos = performance_df['r2'].mean()

print(f"\nPosition-Specific Models (Average):")
print(f"  RMSE: {avg_rmse_pos:.3f}")
print(f"  MAE: {avg_mae_pos:.3f}")
print(f"  R²: {avg_r2_pos:.3f}")

print(f"\nImprovement with Position-Specific Models:")
print(f"  RMSE: {((np.sqrt(mse_uni) - avg_rmse_pos) / np.sqrt(mse_uni) * 100):.1f}%")
print(f"  MAE: {((mae_uni - avg_mae_pos) / mae_uni * 100):.1f}%")
print(f"  R²: {((avg_r2_pos - r2_uni) / abs(r2_uni) * 100):.1f}%")

## Example: Making Predictions with Position-Specific Models

Demonstrate how to use the trained models for predictions on new data.

In [ ]:
# Example: Predict for the most recent gameweek
latest_season = df_main['season'].max()
latest_gw = df_main[df_main['season'] == latest_season]['GW'].max()

print(f"Making predictions for Season: {latest_season}, GW: {latest_gw}")

# Filter data for the latest gameweek
latest_data = df_main[(df_main['season'] == latest_season) & (df_main['GW'] == latest_gw)].copy()

# Make predictions for each position
all_predictions = []

for position in ['GK', 'DEF', 'MID', 'FWD']:
    pos_data = latest_data[latest_data['position_short'] == position].copy()
    
    if len(pos_data) > 0:
        try:
            predictions = position_models.predict(pos_data, position)
            pos_data['predicted_points'] = predictions
            all_predictions.append(pos_data)
            print(f"  {position}: {len(pos_data)} players predicted")
        except Exception as e:
            print(f"  {position}: Error - {str(e)}")

# Combine predictions
if all_predictions:
    predictions_df = pd.concat(all_predictions, ignore_index=True)
    
    # Show top predicted players by position
    print(f"\n{'='*80}")
    print("TOP 5 PREDICTED PLAYERS BY POSITION")
    print('='*80)
    
    for position in ['GK', 'DEF', 'MID', 'FWD']:
        pos_pred = predictions_df[predictions_df['position_short'] == position].copy()
        if len(pos_pred) > 0:
            pos_pred = pos_pred.sort_values('predicted_points', ascending=False).head(5)
            print(f"\n{position}:")
            for idx, row in pos_pred.iterrows():
                player_name = row.get('name', 'Unknown')
                team = row.get('team', 'Unknown')
                pred_pts = row['predicted_points']
                actual_pts = row.get('total_points', 0)
                print(f"  {player_name:30s} ({team:15s}) - Pred: {pred_pts:.1f}, Actual: {actual_pts:.1f}")
else:
    print("No predictions generated")

## Summary: Position-Specific Model Integration

**What was integrated:**

✅ **Section 2: Fixture Difficulty Data**
- Loads the merged and standardized dataset from feature engineering

✅ **Section 3: Position-Specific Feature Engineering**
- Defines custom feature sets for each position (GK, DEF, MID, FWD)
- GK: Focus on saves, clean sheets, opponent attacking strength
- DEF: Clean sheets, tackles, defensive contribution, opponent attack
- MID: Goals, assists, creativity, threat, influence, opponent defense
- FWD: Goals, assists, threat, creativity, opponent defense

✅ **Section 4: Hyperparameter Tuning**
- GridSearchCV and RandomizedSearchCV functions
- Parameter grids for RF, GB, XGBoost, LightGBM, Ridge, ElasticNet

✅ **Section 5: Ensemble Methods**
- Stacking ensemble with multiple base models
- Voting ensemble for averaging predictions
- Custom BlendingEnsemble class for advanced blending

✅ **Section 6: Position-Specific Model Training**
- Separate models for each position using stacking ensembles
- Automated training, evaluation, and saving
- Model comparison against unified single model

**Key Features:**
- Uses the already merged and standardized data from final.ipynb
- Leverages rolling averages and opponent strength features
- Position-specific models capture unique scoring patterns
- Models saved to `models/` directory for future use

**Next Steps:**
- Run the training cells to train position-specific models
- Use the trained models for weekly predictions
- Compare performance against baseline models
- Fine-tune hyperparameters for specific positions if needed

## Section 7: Weekly Predictions Pipeline

Automated predictions pipeline that fetches live data from FPL API and generates gameweek predictions.

In [ ]:
import requests
import json
from datetime import datetime

class WeeklyPredictionsPipeline:
    """Automated weekly predictions with data refresh from FPL API"""
    
    FPL_API_BASE = "https://fantasy.premierleague.com/api/"
    
    def __init__(self, position_models):
        self.position_models = position_models
        self.current_gw = None
        self.predictions_history = []
        
    def fetch_bootstrap_data(self):
        """Fetch main FPL bootstrap data"""
        try:
            response = requests.get(f"{self.FPL_API_BASE}bootstrap-static/", timeout=10)
            response.raise_for_status()
            return response.json()
        except requests.RequestException as e:
            print(f"Error fetching bootstrap data: {e}")
            return None
    
    def fetch_fixtures(self, gw=None):
        """Fetch fixtures data"""
        try:
            url = f"{self.FPL_API_BASE}fixtures/"
            if gw:
                url += f"?event={gw}"
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            return response.json()
        except requests.RequestException as e:
            print(f"Error fetching fixtures: {e}")
            return None
    
    def get_current_gameweek(self, bootstrap_data):
        """Determine current gameweek"""
        events = bootstrap_data.get('events', [])
        for event in events:
            if event.get('is_current'):
                return event['id']
            if event.get('is_next'):
                return event['id']
        return 1
    
    def prepare_player_features(self, player, bootstrap_data, fixtures):
        """Prepare features for a single player prediction"""
        
        teams = {t['id']: t for t in bootstrap_data['teams']}
        
        # Get player's team fixtures
        team_id = player['team']
        next_fixture = None
        
        for fixture in fixtures:
            if fixture['team_h'] == team_id or fixture['team_a'] == team_id:
                if not fixture.get('finished'):
                    next_fixture = fixture
                    break
        
        if not next_fixture:
            return None
        
        is_home = next_fixture['team_h'] == team_id
        opponent_id = next_fixture['team_a'] if is_home else next_fixture['team_h']
        fdr = next_fixture['team_h_difficulty'] if is_home else next_fixture['team_a_difficulty']
        
        # Create feature dict with basic API features
        features = {
            'value': player['now_cost'] / 10,
            'is_home': 1 if is_home else 0,
            'season_progress': self.current_gw / 38,
            'match_number': self.current_gw,
            'total_points_rolling_3': float(player.get('form', 0)),
            'total_points_rolling_5': float(player.get('points_per_game', 0)),
            'total_points_rolling_10': float(player.get('points_per_game', 0)),
            'minutes_rolling_3': player.get('minutes', 0) / max(1, self.current_gw),
            'minutes_rolling_5': player.get('minutes', 0) / max(1, self.current_gw),
            'minutes_rolling_10': player.get('minutes', 0) / max(1, self.current_gw),
            'bps_rolling_3': player.get('bps', 0) / max(1, self.current_gw),
            'bps_rolling_5': player.get('bps', 0) / max(1, self.current_gw),
            'bps_rolling_10': player.get('bps', 0) / max(1, self.current_gw),
            'ict_index_rolling_3': float(player.get('ict_index', 0)) / 10,
            'ict_index_rolling_5': float(player.get('ict_index', 0)) / 10,
            'ict_index_rolling_10': float(player.get('ict_index', 0)) / 10,
            'creativity_rolling_3': float(player.get('creativity', 0)) / 10,
            'creativity_rolling_5': float(player.get('creativity', 0)) / 10,
            'creativity_rolling_10': float(player.get('creativity', 0)) / 10,
            'threat_rolling_3': float(player.get('threat', 0)) / 10,
            'threat_rolling_5': float(player.get('threat', 0)) / 10,
            'threat_rolling_10': float(player.get('threat', 0)) / 10,
            'influence_rolling_3': float(player.get('influence', 0)) / 10,
            'influence_rolling_5': float(player.get('influence', 0)) / 10,
            'influence_rolling_10': float(player.get('influence', 0)) / 10,
            'goals_scored_rolling_3': player.get('goals_scored', 0) / max(1, self.current_gw),
            'goals_scored_rolling_5': player.get('goals_scored', 0) / max(1, self.current_gw),
            'goals_scored_rolling_10': player.get('goals_scored', 0) / max(1, self.current_gw),
            'assists_rolling_3': player.get('assists', 0) / max(1, self.current_gw),
            'assists_rolling_5': player.get('assists', 0) / max(1, self.current_gw),
            'assists_rolling_10': player.get('assists', 0) / max(1, self.current_gw),
            'clean_sheets_rolling_3': player.get('clean_sheets', 0) / max(1, self.current_gw),
            'clean_sheets_rolling_5': player.get('clean_sheets', 0) / max(1, self.current_gw),
            'clean_sheets_rolling_10': player.get('clean_sheets', 0) / max(1, self.current_gw),
            'saves_rolling_3': player.get('saves', 0) / max(1, self.current_gw),
            'saves_rolling_5': player.get('saves', 0) / max(1, self.current_gw),
            'saves_rolling_10': player.get('saves', 0) / max(1, self.current_gw),
            # Opponent features - approximate
            'opponent_strength_attack': fdr,
            'opponent_strength_defense': fdr,
            'opponent_form': 3.0,
            'attacking_advantage': 5.0 - fdr if is_home else 3.0 - fdr,
            'defensive_advantage': 5.0 - fdr if is_home else 3.0 - fdr,
            'tackles_rolling_3': 0,
            'tackles_rolling_5': 0
        }
        
        return features, is_home, opponent_id, fdr
    
    def generate_predictions(self, top_n=50):
        """Generate predictions for all players"""
        
        # Check if models are trained
        if not self.position_models.models:
            print("⚠️ No models have been trained yet!")
            return None
        
        print("Fetching latest FPL data...")
        bootstrap_data = self.fetch_bootstrap_data()
        
        if not bootstrap_data:
            print("Failed to fetch FPL data.")
            return None
        
        self.current_gw = self.get_current_gameweek(bootstrap_data)
        print(f"Current/Next Gameweek: {self.current_gw}")
        
        fixtures = self.fetch_fixtures(self.current_gw)
        if not fixtures:
            print("Failed to fetch fixtures.")
            return None
        
        players = bootstrap_data['elements']
        teams = {t['id']: t['name'] for t in bootstrap_data['teams']}
        positions = {1: 'GK', 2: 'DEF', 3: 'MID', 4: 'FWD'}
        
        predictions = []
        
        print(f"Processing {len(players)} players...")
        
        for player in players:
            # Skip unavailable players
            if player.get('status') in ['i', 'u', 's']:
                continue
            
            position = positions.get(player['element_type'], 'MID')
            
            if position not in self.position_models.models:
                continue
            
            feature_result = self.prepare_player_features(player, bootstrap_data, fixtures)
            if not feature_result:
                continue
            
            features, is_home, opponent_id, fdr = feature_result
            
            # Get features used during training
            required_features = self.position_models.actual_features.get(position, COMMON_FEATURES)
            
            # Create feature vector
            feature_vector = [features.get(f, 0) for f in required_features]
            
            # Make prediction
            try:
                scaler = self.position_models.scalers[position]
                model = self.position_models.models[position]
                
                X = np.array(feature_vector).reshape(1, -1)
                X_scaled = scaler.transform(X)
                predicted_points = model.predict(X_scaled)[0]
                
                predictions.append({
                    'player_id': player['id'],
                    'name': player['web_name'],
                    'team': teams.get(player['team'], 'Unknown'),
                    'position': position,
                    'price': player['now_cost'] / 10,
                    'predicted_points': round(predicted_points, 2),
                    'opponent': teams.get(opponent_id, 'Unknown'),
                    'is_home': is_home,
                    'fdr': fdr,
                    'form': player.get('form', 0),
                    'selected_by': player.get('selected_by_percent', 0),
                    'value': round(predicted_points / (player['now_cost'] / 10), 2)
                })
            except Exception as e:
                continue
        
        if len(predictions) == 0:
            print("\n⚠️ No predictions were generated!")
            return None
        
        # Sort by predicted points
        predictions_df = pd.DataFrame(predictions)
        predictions_df = predictions_df.sort_values('predicted_points', ascending=False)
        
        # Store in history
        self.predictions_history.append({
            'gameweek': self.current_gw,
            'timestamp': datetime.now().isoformat(),
            'predictions': predictions_df.to_dict('records')
        })
        
        print(f"\nGenerated {len(predictions_df)} predictions")
        
        return predictions_df

print("WeeklyPredictionsPipeline class defined!")

In [ ]:
# Initialize pipeline and generate predictions
pipeline = WeeklyPredictionsPipeline(position_models)
weekly_predictions = pipeline.generate_predictions(top_n=30)

if weekly_predictions is not None:
    print("\n" + "="*80)
    print("TOP 30 PREDICTED PLAYERS FOR NEXT GAMEWEEK")
    print("="*80)
    display_cols = ['name', 'team', 'position', 'price', 'predicted_points', 
                    'opponent', 'is_home', 'fdr', 'value']
    print(weekly_predictions[display_cols].head(30).to_string(index=False))

print("Weekly predictions pipeline ready")

## Section 8: Captain Selection Model

Optimize captain selection for maximum expected points using ownership analysis and differential scoring.

In [ ]:
class CaptainSelector:
    """Optimize captain selection for maximum expected points"""
    
    def __init__(self, predictions_pipeline):
        self.pipeline = predictions_pipeline
        
    def get_captain_recommendations(self, team_player_ids, predictions_df, 
                                     consider_ownership=True, differential_threshold=10):
        """Get captain recommendations for a specific team"""
        
        # Filter predictions to team players
        team_predictions = predictions_df[predictions_df['player_id'].isin(team_player_ids)].copy()
        
        if len(team_predictions) == 0:
            print("No matching players found in predictions")
            return None
        
        # Calculate captain score (2x predicted points)
        team_predictions['captain_points'] = team_predictions['predicted_points'] * 2
        
        # Calculate differential score (higher for lower ownership)
        team_predictions['ownership'] = pd.to_numeric(team_predictions['selected_by'], errors='coerce').fillna(50)
        team_predictions['differential_score'] = (
            team_predictions['predicted_points'] * 
            (1 + (100 - team_predictions['ownership']) / 100)
        )
        
        # Identify differentials
        team_predictions['is_differential'] = team_predictions['ownership'] < differential_threshold
        
        # Calculate composite score
        if consider_ownership:
            team_predictions['composite_score'] = (
                team_predictions['predicted_points'] * 0.7 +
                team_predictions['differential_score'] * 0.3
            )
        else:
            team_predictions['composite_score'] = team_predictions['predicted_points']
        
        # Sort by composite score
        team_predictions = team_predictions.sort_values('composite_score', ascending=False)
        
        return team_predictions[['name', 'team', 'position', 'predicted_points', 
                                  'captain_points', 'ownership', 'differential_score',
                                  'is_differential', 'composite_score', 'fdr', 'is_home']]
    
    def analyze_captain_variance(self, predictions_df, n_simulations=1000):
        """Monte Carlo simulation to assess captain pick risk"""
        
        top_captains = predictions_df.nlargest(10, 'predicted_points').copy()
        
        # Assume points follow a distribution around predicted value
        results = []
        
        for _, player in top_captains.iterrows():
            mean_pts = player['predicted_points']
            # Standard deviation based on form variability
            std_pts = max(1, mean_pts * 0.4)  # ~40% coefficient of variation
            
            simulated_points = np.random.normal(mean_pts, std_pts, n_simulations)
            simulated_captain_points = simulated_points * 2
            
            results.append({
                'name': player['name'],
                'predicted': mean_pts,
                'captain_expected': mean_pts * 2,
                'captain_median': np.median(simulated_captain_points),
                'captain_p25': np.percentile(simulated_captain_points, 25),
                'captain_p75': np.percentile(simulated_captain_points, 75),
                'haul_prob': np.mean(simulated_captain_points >= 14),  # Prob of 7+ points
                'blank_prob': np.mean(simulated_captain_points <= 4)   # Prob of 2 or less
            })
        
        return pd.DataFrame(results)

print("CaptainSelector class defined!")

## Section 9: Chip Strategy Optimization

Optimize timing of FPL chips: Wildcard, Bench Boost, Triple Captain, and Free Hit based on fixture analysis.

In [ ]:
class ChipStrategyOptimizer:
    """Optimize timing of FPL chips: Wildcard, Bench Boost, Triple Captain, Free Hit"""
    
    def __init__(self, fixtures_data):
        self.fixtures = fixtures_data
        
    def calculate_fixture_difficulty_rating(self, team_id, gw_range):
        """Calculate average FDR for a team over a gameweek range"""
        fdrs = []
        
        for _, fixture in self.fixtures.iterrows():
            gw = fixture.get('event', fixture.get('GW', 0))
            if gw < gw_range[0] or gw > gw_range[1]:
                continue
            
            team_h = fixture.get('team_h', 0)
            team_a = fixture.get('team_a', 0)
            
            if team_h == team_id:
                fdrs.append(fixture.get('team_h_difficulty', 3))
            elif team_a == team_id:
                fdrs.append(fixture.get('team_a_difficulty', 3))
        
        return np.mean(fdrs) if fdrs else 3
    
    def find_best_wildcard_timing(self, current_gw, remaining_gws=5):
        """Find optimal gameweek to use Wildcard based on fixture swings"""
        
        wildcard_scores = []
        
        # Get unique teams from fixtures
        teams = set()
        for _, fixture in self.fixtures.iterrows():
            teams.add(fixture.get('team_h', 0))
            teams.add(fixture.get('team_a', 0))
        teams = [t for t in teams if t > 0]
        
        for target_gw in range(current_gw, min(current_gw + 10, 39)):
            # Calculate fixture difficulty for next N GWs after wildcard
            future_range = (target_gw, min(target_gw + remaining_gws, 38))
            
            team_scores = []
            for team_id in teams:
                avg_fdr = self.calculate_fixture_difficulty_rating(team_id, future_range)
                team_scores.append({
                    'team_id': team_id,
                    'avg_fdr': avg_fdr
                })
            
            team_scores_df = pd.DataFrame(team_scores)
            
            # Wildcard value = number of teams with good fixtures (FDR <= 2.5)
            good_fixture_teams = len(team_scores_df[team_scores_df['avg_fdr'] <= 2.5])
            
            # Best average FDR from top 5 teams
            avg_best_fdr = team_scores_df.nsmallest(5, 'avg_fdr')['avg_fdr'].mean() if len(team_scores_df) >= 5 else 3
            
            wildcard_scores.append({
                'gameweek': target_gw,
                'good_fixture_teams': good_fixture_teams,
                'avg_best_fdr': avg_best_fdr
            })
        
        return pd.DataFrame(wildcard_scores).sort_values('avg_best_fdr')
    
    def find_best_bench_boost_timing(self, current_gw):
        """Find optimal GW for Bench Boost (maximize total squad points)"""
        
        gw_scores = []
        
        for gw in range(current_gw, 39):
            gw_col = 'event' if 'event' in self.fixtures.columns else 'GW'
            gw_fixtures = self.fixtures[self.fixtures[gw_col] == gw]
            
            # Count games (DGW detection)
            total_games = len(gw_fixtures)
            
            # Average fixture difficulty
            avg_home_fdr = gw_fixtures['team_h_difficulty'].mean() if len(gw_fixtures) > 0 and 'team_h_difficulty' in gw_fixtures.columns else 3
            avg_away_fdr = gw_fixtures['team_a_difficulty'].mean() if len(gw_fixtures) > 0 and 'team_a_difficulty' in gw_fixtures.columns else 3
            
            # Bench boost score (prefer DGWs and easy fixtures)
            dgw_bonus = 1.5 if total_games > 10 else 1.0
            bb_score = (10 - (avg_home_fdr + avg_away_fdr) / 2) * dgw_bonus
            
            gw_scores.append({
                'gameweek': gw,
                'total_games': total_games,
                'is_dgw': total_games > 10,
                'avg_fdr': (avg_home_fdr + avg_away_fdr) / 2,
                'bb_score': bb_score
            })
        
        return pd.DataFrame(gw_scores).sort_values('bb_score', ascending=False)
    
    def find_best_free_hit_timing(self, current_gw):
        """Find optimal GW for Free Hit (maximize single GW team)"""
        
        fh_scores = []
        
        for gw in range(current_gw, 39):
            gw_col = 'event' if 'event' in self.fixtures.columns else 'GW'
            gw_fixtures = self.fixtures[self.fixtures[gw_col] == gw]
            
            # Count teams playing
            teams_h = set(gw_fixtures['team_h'].tolist()) if 'team_h' in gw_fixtures.columns else set()
            teams_a = set(gw_fixtures['team_a'].tolist()) if 'team_a' in gw_fixtures.columns else set()
            teams_playing = teams_h.union(teams_a)
            teams_with_fixtures = len(teams_playing)
            
            # Free Hit is best in blank gameweeks (BGW) or DGWs
            is_bgw = teams_with_fixtures < 15
            is_dgw = len(gw_fixtures) > 10
            
            # Calculate fixture favorability
            easy_home_fixtures = len(gw_fixtures[gw_fixtures['team_h_difficulty'] <= 2]) if 'team_h_difficulty' in gw_fixtures.columns else 0
            easy_away_fixtures = len(gw_fixtures[gw_fixtures['team_a_difficulty'] <= 2]) if 'team_a_difficulty' in gw_fixtures.columns else 0
            
            fh_score = easy_home_fixtures + easy_away_fixtures
            if is_bgw:
                fh_score += 10  # Bonus for BGW (limited options)
            if is_dgw:
                fh_score += 5   # Bonus for DGW
            
            fh_scores.append({
                'gameweek': gw,
                'teams_playing': teams_with_fixtures,
                'is_bgw': is_bgw,
                'is_dgw': is_dgw,
                'easy_fixtures': easy_home_fixtures + easy_away_fixtures,
                'fh_score': fh_score
            })
        
        return pd.DataFrame(fh_scores).sort_values('fh_score', ascending=False)

print("ChipStrategyOptimizer class defined!")

In [ ]:
# Example usage of chip optimizer
# Load fixtures for current season
fixtures_df = pd.read_csv('data/2024-25/fixtures.csv')
chip_optimizer = ChipStrategyOptimizer(fixtures_df)
current_gw = 20
print("="*70)
print("CHIP STRATEGY RECOMMENDATIONS")
print("="*70)

# Wildcard timing
print("\n📋 WILDCARD TIMING ANALYSIS:")
wc_analysis = chip_optimizer.find_best_wildcard_timing(current_gw)
print(wc_analysis.head(5).to_string(index=False))

# Bench Boost timing
print("\n🪑 BENCH BOOST TIMING ANALYSIS:")
bb_analysis = chip_optimizer.find_best_bench_boost_timing(current_gw)
print(bb_analysis.head(5).to_string(index=False))

# Free Hit timing
print("\n🎯 FREE HIT TIMING ANALYSIS:")
fh_analysis = chip_optimizer.find_best_free_hit_timing(current_gw)
print(fh_analysis.head(5).to_string(index=False))

print("Chip strategy optimizer ready")

## Section 10: Historical Backtesting Engine

Simulate how position-specific models would have performed in past seasons to validate strategy effectiveness.

In [ ]:
class HistoricalBacktester:
    """Backtest FPL strategies across historical seasons - Optimized Version"""
    
    BUDGET = 100.0
    SQUAD_SIZE = 15
    STARTING_XI = 11
    MAX_PER_TEAM = 3
    
    POSITION_LIMITS = {
        'GK': (2, 2),
        'DEF': (5, 5),
        'MID': (5, 5),
        'FWD': (3, 3)
    }
    
    def __init__(self, historical_data, position_models=None):
        self.data = historical_data
        self.position_models = position_models
        self.results = []
    
    def _apply_heuristic_predictions(self, gw_data):
        """Balanced heuristic predictions - proven best approach"""
        gw_data = gw_data.copy()
        
        # Start with position baseline
        position_baseline = {'GK': 3.5, 'DEF': 4.0, 'MID': 4.5, 'FWD': 4.5}
        gw_data['predicted_points'] = gw_data['position'].map(position_baseline).fillna(3.5)
        
        # 1. PRIMARY: Rolling average is best predictor
        if 'total_points_rolling_3' in gw_data.columns:
            valid = gw_data['total_points_rolling_3'].notna() & (gw_data['total_points_rolling_3'] > 0)
            # Use 80% rolling_3 + 20% rolling_5 for stability
            if 'total_points_rolling_5' in gw_data.columns:
                r3 = gw_data.loc[valid, 'total_points_rolling_3'].fillna(0)
                r5 = gw_data.loc[valid, 'total_points_rolling_5'].fillna(r3)
                gw_data.loc[valid, 'predicted_points'] = 0.8 * r3 + 0.2 * r5
            else:
                gw_data.loc[valid, 'predicted_points'] = gw_data.loc[valid, 'total_points_rolling_3']
        
        # 2. ICT Index - FPL's underlying quality metric (strong correlation)
        if 'ict_index_rolling_3' in gw_data.columns:
            valid_ict = gw_data['ict_index_rolling_3'].notna() & (gw_data['ict_index_rolling_3'] > 2)
            if valid_ict.sum() > 0:
                # ICT correlates ~0.5 with points
                ict_pred = gw_data.loc[valid_ict, 'ict_index_rolling_3'] * 0.45
                gw_data.loc[valid_ict, 'predicted_points'] = (
                    0.6 * gw_data.loc[valid_ict, 'predicted_points'] + 0.4 * ict_pred
                )
        
        # 3. CRITICAL: Minutes filter (non-playing players = 0 points)
        if 'minutes_rolling_3' in gw_data.columns:
            low_minutes = gw_data['minutes_rolling_3'].fillna(0) < 45
            gw_data.loc[low_minutes, 'predicted_points'] *= 0.2
        
        # 4. Price as quality proxy for premium players
        if 'price' in gw_data.columns:
            # Premium boost: expensive players usually deliver
            premium = gw_data['price'] >= 10.0
            gw_data.loc[premium, 'predicted_points'] *= 1.15
            
            # Budget penalty: very cheap players usually don't score much
            budget_players = gw_data['price'] <= 4.5
            gw_data.loc[budget_players, 'predicted_points'] *= 0.85
        
        # 5. Attacking returns boost (goals = 4-5 pts, assists = 3 pts)
        if 'goals_scored_rolling_3' in gw_data.columns:
            valid = gw_data['goals_scored_rolling_3'].notna()
            # Goal scoring rate * expected points per goal
            mids_fwds = gw_data['position'].isin(['MID', 'FWD'])
            gw_data.loc[valid & mids_fwds, 'predicted_points'] += (
                gw_data.loc[valid & mids_fwds, 'goals_scored_rolling_3'] * 2.0
            )
        
        if 'assists_rolling_3' in gw_data.columns:
            valid = gw_data['assists_rolling_3'].notna()
            gw_data.loc[valid, 'predicted_points'] += gw_data.loc[valid, 'assists_rolling_3'] * 1.5
        
        # 6. Bonus magnet bonus
        if 'bonus_rolling_3' in gw_data.columns:
            valid = gw_data['bonus_rolling_3'].notna()
            gw_data.loc[valid, 'predicted_points'] += gw_data.loc[valid, 'bonus_rolling_3'] * 0.8
        
        # 7. Clean sheet probability for defensive players
        if 'clean_sheets_rolling_3' in gw_data.columns:
            def_gk = gw_data['position'].isin(['GK', 'DEF'])
            valid = def_gk & gw_data['clean_sheets_rolling_3'].notna()
            # Clean sheet = 4 points for DEF/GK
            gw_data.loc[valid, 'predicted_points'] += gw_data.loc[valid, 'clean_sheets_rolling_3'] * 3.0
        
        # 8. GK saves bonus
        if 'saves_rolling_3' in gw_data.columns:
            gks = gw_data['position'] == 'GK'
            valid = gks & gw_data['saves_rolling_3'].notna()
            gw_data.loc[valid, 'predicted_points'] += gw_data.loc[valid, 'saves_rolling_3'] * 0.3
        
        # Clip to reasonable range
        gw_data['predicted_points'] = gw_data['predicted_points'].clip(lower=0.5, upper=25.0)
        
        return gw_data
        
    def select_team(self, gw_predictions, budget=100.0):
        """Robust team selection that always fills 15 slots within budget"""
        
        selected_players = []
        remaining_budget = budget
        team_counts = {}
        position_counts = {'GK': 0, 'DEF': 0, 'MID': 0, 'FWD': 0}
        
        gw_predictions = gw_predictions.copy()
        
        # Calculate scores - emphasize value more to stay within budget
        gw_predictions['value'] = gw_predictions['predicted_points'] / gw_predictions['price'].clip(lower=4.0)
        max_pred = gw_predictions['predicted_points'].max()
        max_val = gw_predictions['value'].max()
        if max_val > 0 and max_pred > 0:
            # More balanced: 60% value, 40% absolute
            gw_predictions['score'] = (
                0.6 * gw_predictions['value'] / max_val +
                0.4 * gw_predictions['predicted_points'] / max_pred
            )
        else:
            gw_predictions['score'] = gw_predictions['predicted_points']
        
        # Calculate average price per position needed
        avg_budget_per_player = budget / 15  # ~6.67
        
        # First: Fill each position with best value players
        for pos, (min_req, max_req) in self.POSITION_LIMITS.items():
            pos_players = gw_predictions[gw_predictions['position'] == pos].copy()
            
            # Sort by score (value-weighted)
            pos_players = pos_players.sort_values('score', ascending=False)
            
            added = 0
            for _, player in pos_players.iterrows():
                if added >= min_req:
                    break
                team = player['team']
                
                # Check if affordable
                players_remaining = 15 - len(selected_players) - 1  # exclude current
                min_budget_needed = players_remaining * 4.0  # min price assumption
                affordable = (player['price'] <= remaining_budget - min_budget_needed)
                
                if affordable and team_counts.get(team, 0) < self.MAX_PER_TEAM:
                    selected_players.append(player)
                    remaining_budget -= player['price']
                    team_counts[team] = team_counts.get(team, 0) + 1
                    position_counts[pos] = position_counts.get(pos, 0) + 1
                    added += 1
            
            # If couldn't fill minimums with best value, use cheapest players
            if added < min_req:
                pos_players_cheap = gw_predictions[
                    (gw_predictions['position'] == pos) & 
                    (~gw_predictions['name'].isin([p['name'] for p in selected_players]))
                ].sort_values('price', ascending=True)
                
                for _, player in pos_players_cheap.iterrows():
                    if added >= min_req:
                        break
                    team = player['team']
                    
                    players_remaining = 15 - len(selected_players) - 1
                    min_budget_needed = max(0, players_remaining * 4.0)
                    affordable = (player['price'] <= remaining_budget - min_budget_needed)
                    
                    if affordable and team_counts.get(team, 0) < self.MAX_PER_TEAM:
                        selected_players.append(player)
                        remaining_budget -= player['price']
                        team_counts[team] = team_counts.get(team, 0) + 1
                        position_counts[pos] = position_counts.get(pos, 0) + 1
                        added += 1
        
        # Second: Fill remaining slots with best affordable players
        selected_names = [p['name'] for p in selected_players]
        remaining = gw_predictions[~gw_predictions['name'].isin(selected_names)]
        remaining = remaining.sort_values('score', ascending=False)
        
        for _, player in remaining.iterrows():
            if len(selected_players) >= self.SQUAD_SIZE:
                break
            
            position = player['position']
            team = player['team']
            price = player['price']
            
            # Budget check with safety margin
            slots_left = self.SQUAD_SIZE - len(selected_players) - 1
            min_needed = slots_left * 4.0
            
            if price > remaining_budget - min_needed:
                continue
            if team_counts.get(team, 0) >= self.MAX_PER_TEAM:
                continue
            if position_counts.get(position, 0) >= self.POSITION_LIMITS.get(position, (0,5))[1]:
                continue
            
            selected_players.append(player)
            remaining_budget -= price
            team_counts[team] = team_counts.get(team, 0) + 1
            position_counts[position] = position_counts.get(position, 0) + 1
        
        # Third: Fill any remaining slots with cheapest affordable players
        if len(selected_players) < self.SQUAD_SIZE:
            selected_names = [p['name'] for p in selected_players]
            cheapest = gw_predictions[~gw_predictions['name'].isin(selected_names)]
            cheapest = cheapest.sort_values('price', ascending=True)
            
            for _, player in cheapest.iterrows():
                if len(selected_players) >= self.SQUAD_SIZE:
                    break
                
                position = player['position']
                team = player['team']
                
                if team_counts.get(team, 0) >= self.MAX_PER_TEAM:
                    continue
                if position_counts.get(position, 0) >= self.POSITION_LIMITS.get(position, (0,5))[1]:
                    continue
                
                # Only add if we can afford it
                if player['price'] <= remaining_budget:
                    selected_players.append(player)
                    remaining_budget -= player['price']
                    team_counts[team] = team_counts.get(team, 0) + 1
                    position_counts[position] = position_counts.get(position, 0) + 1
        
        return pd.DataFrame(selected_players), remaining_budget
    
    def select_starting_xi(self, squad):
        """Select best starting XI with proper formation constraints"""
        
        starting_xi = []
        
        # Always start best GK
        gks = squad[squad['position'] == 'GK'].nlargest(1, 'predicted_points')
        starting_xi.extend(gks.to_dict('records'))
        
        # Sort outfield by predicted points
        outfield = squad[squad['position'] != 'GK'].sort_values('predicted_points', ascending=False)
        
        position_counts = {'DEF': 0, 'MID': 0, 'FWD': 0}
        min_requirements = {'DEF': 3, 'MID': 2, 'FWD': 1}
        max_limits = {'DEF': 5, 'MID': 5, 'FWD': 3}
        
        # First ensure minimums
        for pos in ['DEF', 'MID', 'FWD']:
            pos_players = outfield[outfield['position'] == pos].head(min_requirements[pos])
            for _, p in pos_players.iterrows():
                if p['name'] not in [x['name'] for x in starting_xi]:
                    starting_xi.append(p.to_dict())
                    position_counts[pos] += 1
        
        # Fill remaining with highest predicted
        for _, player in outfield.iterrows():
            if len(starting_xi) >= 11:
                break
            if player['name'] in [x['name'] for x in starting_xi]:
                continue
            
            pos = player['position']
            if position_counts[pos] >= max_limits[pos]:
                continue
            
            starting_xi.append(player.to_dict())
            position_counts[pos] += 1
        
        return pd.DataFrame(starting_xi)
    
    def calculate_gw_points(self, starting_xi, captain_id, vice_captain_id, actual_data):
        """Calculate actual points with captain and vice-captain logic"""
        
        total_points = 0
        captain_played = False
        captain_points = 0
        vice_captain_points = 0
        
        for _, player in starting_xi.iterrows():
            points = float(player.get('total_points', 0)) if pd.notna(player.get('total_points', 0)) else 0
            
            is_captain = (player.get('element') == captain_id or player.get('name') == captain_id)
            is_vice = (player.get('element') == vice_captain_id or player.get('name') == vice_captain_id)
            
            if is_captain:
                captain_points = points
                captain_played = points > 0 or player.get('minutes', 0) > 0
            elif is_vice:
                vice_captain_points = points
            
            total_points += points
        
        # Captain gets double - if captain didn't play, vice captain gets double
        if captain_played:
            total_points += captain_points  # Add captain bonus
        else:
            total_points += vice_captain_points  # Vice captain becomes captain
        
        return total_points
    
    def backtest_season(self, season, strategy='predicted_points'):
        """Backtest a full season with improved strategy"""
        
        season_data = self.data[self.data['season_x'] == season].copy()
        
        if len(season_data) == 0:
            print(f"No data available for season {season}")
            return None
        
        gameweeks = sorted(season_data['GW'].unique())
        
        total_points = 0
        gw_results = []
        current_squad = None
        
        for gw in gameweeks:
            gw_data = season_data[season_data['GW'] == gw].copy()
            
            if len(gw_data) < 50:
                continue
            
            # Prepare columns
            gw_data['team'] = gw_data.get('team_x', gw_data.get('team', 'Unknown'))
            position_map = {'GKP': 'GK', 'GK': 'GK', 'DEF': 'DEF', 'MID': 'MID', 'FWD': 'FWD'}
            gw_data['position'] = gw_data['position'].map(position_map).fillna('MID')
            
            # Price calculation
            if 'value' in gw_data.columns:
                gw_data['price'] = gw_data['value'].fillna(50) / 10
            elif 'now_cost' in gw_data.columns:
                gw_data['price'] = gw_data['now_cost'].fillna(50) / 10
            else:
                gw_data['price'] = 5.0
            
            # Apply enhanced heuristic predictions
            gw_data = self._apply_heuristic_predictions(gw_data)
            
            # Ensure total_points is numeric
            if 'total_points' in gw_data.columns:
                gw_data['total_points'] = pd.to_numeric(gw_data['total_points'], errors='coerce').fillna(0)
            
            gw_data['predicted_points'] = pd.to_numeric(gw_data['predicted_points'], errors='coerce').fillna(2)
            
            # CRITICAL: Deduplicate by player name, keeping the row with best total_points
            # This prevents selecting the same player multiple times
            gw_data = gw_data.sort_values('total_points', ascending=False)
            gw_data = gw_data.drop_duplicates(subset=['name'], keep='first')
            
            # Filter to playing players only (those with reasonable minutes expectation)
            # This helps with budget allocation
            playing_players = gw_data[gw_data['predicted_points'] >= 1.0].copy()
            
            # Select squad with slight budget flexibility
            current_squad, remaining = self.select_team(playing_players, budget=100.0)
            
            # If we couldn't form a squad, try with all players
            if len(current_squad) < 15:
                current_squad, remaining = self.select_team(gw_data, budget=100.0)
            
            if len(current_squad) < 11:
                # Still can't form - skip this GW
                continue
            
            # Select starting XI
            starting_xi = self.select_starting_xi(current_squad)
            
            if len(starting_xi) < 11:
                continue
            
            # Select captain (highest predicted) and vice captain (second highest)
            top_players = starting_xi.nlargest(2, 'predicted_points')
            captain = top_players.iloc[0]
            vice_captain = top_players.iloc[1] if len(top_players) > 1 else captain
            
            captain_id = captain.get('element', captain['name'])
            vice_captain_id = vice_captain.get('element', vice_captain['name'])
            
            # Calculate actual points
            gw_points = self.calculate_gw_points(starting_xi, captain_id, vice_captain_id, gw_data)
            
            total_points += gw_points
            
            gw_results.append({
                'gameweek': gw,
                'points': gw_points,
                'cumulative': total_points,
                'captain': captain['name']
            })
        
        results_df = pd.DataFrame(gw_results)
        
        self.results.append({
            'season': season,
            'total_points': total_points,
            'gameweeks_played': len(gw_results),
            'avg_points_per_gw': total_points / max(1, len(gw_results)),
            'details': results_df
        })
        
        return results_df
    
    def backtest_all_seasons(self):
        """Backtest across all available seasons"""
        
        seasons = self.data['season_x'].unique()
        
        print(f"Backtesting {len(seasons)} seasons...")
        
        for season in sorted(seasons):
            print(f"\nBacktesting {season}...")
            self.backtest_season(season)
        
        return self.get_summary()
    
    def get_summary(self):
        """Get backtest summary"""
        
        summary = []
        for result in self.results:
            summary.append({
                'season': result['season'],
                'total_points': result['total_points'],
                'gameweeks': result['gameweeks_played'],
                'avg_ppg': result['avg_points_per_gw']
            })
        
        return pd.DataFrame(summary)

print("HistoricalBacktester class defined!")

In [ ]:
# Run backtest on historical data
print("="*70)
print("HISTORICAL BACKTESTING")
print("="*70)

backtester = HistoricalBacktester(df_merged, position_models)

# Backtest recent seasons
seasons_to_test = ['2021-22', '2022-23', '2023-24']

for season in seasons_to_test:
    print(f"\nBacktesting {season}...")
    result = backtester.backtest_season(season)
    if result is not None and len(result) > 0:
        print(f"Gameweeks: {len(result)}")
        print(f"Season points: {result['cumulative'].max():.0f}")
        print(f"Average per GW: {result['points'].mean():.1f}")

In [ ]:
# Final performance summary
print("="*70)
print("FINAL BACKTEST RESULTS")
print("="*70)

total_gws = sum(r['gameweeks_played'] for r in backtester.results)
total_pts = sum(r['total_points'] for r in backtester.results)
avg_ppg = total_pts / total_gws if total_gws > 0 else 0

print(f"\n{'Season':<12} {'GWs':>6} {'Points':>8} {'PPG':>8}")
print("-"*36)
for r in backtester.results:
    print(f"{r['season']:<12} {r['gameweeks_played']:>6} {r['total_points']:>8.0f} {r['avg_points_per_gw']:>8.1f}")
print("-"*36)
print(f"{'TOTAL':<12} {total_gws:>6} {total_pts:>8.0f} {avg_ppg:>8.1f}")

# Top 10 captains analysis
print("\n" + "="*70)
print("TOP CAPTAIN PERFORMERS")
print("="*70)

all_gw_data = []
for r in backtester.results:
    if 'details' in r:
        details = r['details'].copy()
        details['season'] = r['season']
        all_gw_data.append(details)

all_results = pd.concat(all_gw_data, ignore_index=True)
captain_stats = all_results.groupby('captain')['points'].agg(['mean', 'count', 'sum']).round(1)
captain_stats = captain_stats[captain_stats['count'] >= 3]  # At least 3 GWs as captain
top_captains = captain_stats.sort_values('mean', ascending=False).head(10)
print(top_captains)

# Best and worst gameweeks
print("\n" + "="*70)
print("BEST GAMEWEEKS")
print("="*70)
best_gws = all_results.nlargest(5, 'points')
print(best_gws[['season', 'gameweek', 'points', 'captain']].to_string(index=False))

In [ ]:
# Display backtest summary
print("\n" + "="*70)
print("BACKTEST SUMMARY")
print("="*70)

summary = backtester.get_summary()
if len(summary) > 0:
    print(summary.to_string(index=False))
    
    print(f"\nOverall Average Points per GW: {summary['avg_ppg'].mean():.1f}")
    print(f"Best Season: {summary.loc[summary['total_points'].idxmax(), 'season']} with {summary['total_points'].max():.0f} points")
    
    # Save results
    summary.to_csv('backtest_summary.csv', index=False)
    print("\nBacktest summary saved to backtest_summary.csv")
    
    # Expected benchmark: Top FPL managers score 2300-2500 points per season
    # Average: ~50-60 points per gameweek
    avg_ppg = summary['avg_ppg'].mean()
    if avg_ppg >= 50:
        print(f"✅ Performance is in good range (50+ PPG is competitive)")
    elif avg_ppg >= 40:
        print(f"⚠️ Performance is moderate (40-50 PPG)")
    else:
        print(f"❌ Performance is below expected - check data/model")
else:
    print("No backtest results available")

In [ ]:
# Visualize backtest results
if len(backtester.results) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Points per gameweek over time
    ax1 = axes[0]
    for result in backtester.results:
        if result['details'] is not None and len(result['details']) > 0:
            ax1.plot(result['details']['gameweek'], result['details']['points'], 
                    label=result['season'], alpha=0.7)
    ax1.set_xlabel('Gameweek')
    ax1.set_ylabel('Points')
    ax1.set_title('Points per Gameweek by Season')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Cumulative points
    ax2 = axes[1]
    for result in backtester.results:
        if result['details'] is not None and len(result['details']) > 0:
            ax2.plot(result['details']['gameweek'], result['details']['cumulative'], 
                    label=result['season'], alpha=0.7)
    ax2.set_xlabel('Gameweek')
    ax2.set_ylabel('Cumulative Points')
    ax2.set_title('Cumulative Points by Season')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No backtest results to visualize")

## Section 11: Save Models and Export Results

Save trained models and export predictions for future use.

In [ ]:
# Save trained models
import os

# Create models directory
models_dir = 'models/'
os.makedirs(models_dir, exist_ok=True)

# Save position-specific models
position_models.save_models(models_dir)
print(f"Models saved to {models_dir}")

print("Model saving functionality ready")

In [ ]:
# Export predictions to CSV

# Export weekly predictions
if 'weekly_predictions' in locals() and weekly_predictions is not None:
    output_file = f'predictions_gw{pipeline.current_gw}.csv'
    weekly_predictions.to_csv(output_file, index=False)
    print(f"Predictions exported to {output_file}")

# Load saved models for future use
position_models_loaded = PositionSpecificModels()
position_models_loaded.load_models('models/')

print("Export functionality ready")